# BioSaccade-PEM: self-auditing foveated active vision (local CPU version)

Complete, resumable experiment notebook for the revised manuscript, adapted for a local Windows laptop **without a GPU** (all neural baselines run on the CPU).

**How to run.** Run the cells from top to bottom. Start with `PROFILE = "smoke"` (a few minutes, synthetic data, checks the code), then `"laptop"` (a pilot on real data), then `"paper"`. Results go to `<RESULTS_DIR>`. Every finished job is checkpointed there, so if the run stops, re-run all cells and it continues where it stopped.

**Datasets (local copies).** DAVIS 2017 trainval 480p (`<DAVIS_ROOT>`), REDS `val_sharp` (`<REDS_ROOT>\val_sharp`, 30 clips x 100 frames) and REDS `train_sharp` (`<REDS_ROOT>\train_sharp`, 240 clips x 100 frames). REDS `train_sharp` replaces Vimeo-90K, whose official download is no longer available. **No model in this study is trained on REDS**: the PEM memory and auditor are fitted only on DAVIS fit sources, the BasicVSR++ checkpoint is trained on the Vimeo-90K training split, and the NTIRE 2025 ESR models are still-image checkpoints. All 270 REDS clips (train + val) are therefore unseen test data; the paper must state this explicitly. REDS train is a separate dataset (`REDS_train`) so its clip IDs (`000`...) never collide with REDS val.

**What the paper profile evaluates.**

| Study | Data | Purpose |
|---|---|---|
| `main` | REDS validation (30 clips, 100 frames) + DAVIS 480p legacy test (40 clips) | Primary evidence: long clips, nominal and drifting sensors, 2 draws, all controlled policies and ablations, plus the secondary variant TwinAudit-strict (κw = 1) |
| `reds_train_full` | All 240 REDS `train_sharp` clips (100 frames) | Scale and generalisation: greedy, previous version, PPAT, TwinAudit |
| `reds_train_detail` | Fixed 60-clip REDS `train_sharp` subset | All controlled policies, 2 draws |
| `neural_native` | REDS val (30) + 10 DAVIS + 20 REDS train, clean MATLAB-bicubic input | Published SR/VSR checkpoints in their native degradation (fair to baselines) |
| `neural_sensor` | Same clips, the paper's noisy sensor | Same baselines in the paper's sensing model |
| `timing` | 10 REDS val clips, run serially | Latency (the only study used for runtime claims) |
| sensitivity | Fixed subsets (15 REDS val + 15 DAVIS + 15 REDS train) | κ, certificate κ, write κ, preview factor/kernel, and an assumption-violating negative control |

**Published baselines.** SPAN-F, NanoSR and SCMSR (NTIRE 2025 efficient SR, official checkpoints pinned by commit and SHA-256), BasicVSR++ (CVPR 2022, official Vimeo-90K BI checkpoint, trained only on the Vimeo-90K training split) in a causal sliding-window mode and a non-causal offline reference, and PPAT (2026). A behavioural sanity gate stops the run if any pretrained baseline fails to beat bicubic upsampling, so a broken baseline cannot enter the tables.

**Post-pilot amendment.** Added after the laptop pilot (local_01_677c295da848) and before the paper run: paired same-backbone comparisons in the neural studies, backbone + TwinAudit gaze (_memory_R9) vs backbone + memory (_memory) and vs backbone + patch (_patch), for PSNR, MSE, audit PQV and write PQV, using the same source-cluster bootstrap, sign-flip test and Holm adjustment (within study and metric). Also added at the same time: a secondary variant PEM_R9_TwinAudit_strict (TwinAudit with write_kappa = 1, displayed as "TwinAudit-strict (κw = 1)") in the main study only. It is a secondary variant, NOT the primary method: TwinAudit with the development-selected write_kappa = 2 remains the primary method and the target of all paired comparisons, and the development selection was not changed. Speed-only changes: THREADS raised from 2 to 8, and neural baseline predictions computed in parallel worker processes (each job computed by the same functions as before; equivalence verified before the paper run). Reproducibility fix: the official SCMSR code samples gumbel_softmax even at inference, so the global PyTorch RNG is now seeded per clip, condition and model before each baseline prediction (and per model in the sanity gate); SCMSR numbers therefore differ slightly from the laptop pilot, where its random draws depended on call order. No method, selection rule or existing statistic was changed.

**Expected cost.** The cell before the run prints the exact plan (number of frame-method evaluations).


## 1. Setup

In [ ]:
import os, sys, subprocess, importlib.metadata
from pathlib import Path

# Local Windows laptop, CPU only (no GPU). Neural baselines run on the CPU.
PROFILE = "smoke"          # "smoke" -> "laptop" -> "paper"
RUN_NEURAL = True          # published SR / video-SR baselines (CPU)
INCLUDE_VIDEO_SR = True    # BasicVSR++ causal + offline (non-causal) reference
DEVICE = "cpu"
WORKERS = 5                # CPU worker processes for the controller/evaluator
THREADS = 8                # PyTorch/OpenCV threads in the main process (sanity gate, timing study); speed only
NEURAL_WORKERS = 3         # parallel neural-baseline processes (~1.5 GB RAM each at 100 frames); speed only
NEURAL_THREADS = 2         # PyTorch threads per neural worker; speed only

# Results, checkpoints, downloaded model checkpoints and evidence (persistent, resumable).
WORKDIR = Path(r"<RESULTS_DIR>")
# Scratch data written by the pipeline (synthetic smoke-test frames).
LOCAL_DATA = WORKDIR / "data_cache"

# EDIT the three <...> placeholders below (and <RESULTS_DIR> above) before running.
# Local dataset copies. REDS train and val stay in separate folders (both start at 000)
# and are registered as separate datasets so their clip IDs never collide.
DATASET_ROOTS = {
    "DAVIS": r"<DAVIS_ROOT>",                  # DAVIS 2017 trainval 480p: JPEGImages/480p/<clip>/*.jpg
    "REDS": r"<REDS_ROOT>\val_sharp",           # REDS val_sharp: 30 clips x 100 frames
    "REDS_train": r"<REDS_ROOT>\train_sharp",  # REDS train_sharp: 240 clips x 100 frames
}
for name, path in DATASET_ROOTS.items():
    if not Path(path).is_dir():
        raise FileNotFoundError(f"{name} not found at {path}")
for path in [WORKDIR, LOCAL_DATA, *DATASET_ROOTS.values()]:
    if "onedrive" in str(path).lower():
        raise ValueError(f"Keep data and results outside OneDrive: {path}")

DOWNLOAD_DATA = False      # datasets are local; never download or substitute data silently
EXPERIMENT_ID = "local_01"
HIGH_RESOLUTION_REDS = False
SAVE_ALL_TRACES = False
EXPORT_FIGURES = True
LINE_DPI = 600             # raster copies; PDF/SVG vector copies are always exported

assert sys.version_info >= (3, 10)

try:
    from packaging.requirements import Requirement
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "packaging>=23"])
    from packaging.requirements import Requirement


def _missing(specs):
    out = []
    for spec in specs:
        req = Requirement(spec)
        try:
            if importlib.metadata.version(req.name) not in req.specifier:
                out.append(spec)
        except importlib.metadata.PackageNotFoundError:
            out.append(spec)
    return out


requirements = [
    "numpy>=1.26,<3",
    "scipy>=1.12,<2",
    "pandas>=2,<4",
    "scikit-learn>=1.4,<2",
    "matplotlib>=3.8,<4",
    "joblib>=1.3,<2",
    "psutil>=5.9,<8",
]
try:
    import cv2
    assert tuple(map(int, cv2.__version__.split(".")[:2])) >= (4, 8)
except (ImportError, AssertionError):
    requirements.append("opencv-python-headless>=4.8,<5")
if RUN_NEURAL:
    requirements.append("einops>=0.7,<1")
missing = _missing(requirements)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
if RUN_NEURAL:
    # CPU-only PyTorch wheels (no CUDA download).
    missing_torch = _missing(["torch>=2.2,<3", "torchvision>=0.17"])
    if missing_torch:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_torch,
                               "--index-url", "https://download.pytorch.org/whl/cpu"])

WORKDIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
SOURCE_DIR = WORKDIR / "src"
SOURCE_DIR.mkdir(exist_ok=True)

import psutil
print("Profile:", PROFILE)
print("Results:", WORKDIR)
print("CPU cores:", os.cpu_count(), "| RAM (GB):", round(psutil.virtual_memory().total / 2**30, 1), "| workers:", WORKERS)
print("Datasets:", DATASET_ROOTS)
if RUN_NEURAL:
    import torch
    print("PyTorch:", torch.__version__, "| device:", DEVICE)


## 2. Memory model

In [ ]:
SOURCE = r'''"""Predictive memory model."""
import hashlib
import numpy as np
import cv2
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor

FEATURES=['log_detail_energy','log_innovation','relative_innovation','log_age',
 'capped_age','flow_inconsistency','flow_speed','log_gradient','log_curvature',
 'log_preview_noise','old_gradient','detail_gradient_ratio','short_change',
 'log_texture','log_anchor_error','relative_anchor_error','cumulative_innovation']

def seed(*parts):
    return int.from_bytes(hashlib.sha256('|'.join(map(str,parts)).encode()).digest()[:4],'little')

def down(x,k):
    h,w,c=x.shape
    return x.reshape(h//k,k,w//k,k,c).mean((1,3))


def _cubic(x):
    ax=np.abs(x); ax2=ax*ax; ax3=ax2*ax
    return ((1.5*ax3-2.5*ax2+1)*(ax<=1)+(-.5*ax3+2.5*ax2-4*ax+2)*((ax>1)&(ax<=2))).astype(np.float64)

def _imresize_weights(in_len,out_len,scale):
    """MATLAB imresize bicubic weights with antialiasing for downscaling (scale<1)."""
    width=4./scale
    x=np.arange(1,out_len+1,dtype=np.float64)
    u=x/scale+.5*(1-1/scale)
    left=np.floor(u-width/2)
    taps=int(np.ceil(width))+2
    idx=left[:,None]+np.arange(taps)[None,:]-1
    w=scale*_cubic(scale*(u[:,None]-idx-1))
    w=w/w.sum(1,keepdims=True)
    # Symmetric boundary handling, as in MATLAB.
    aux=np.concatenate([np.arange(in_len),np.arange(in_len-1,-1,-1)])
    idx=aux[np.mod(idx.astype(int),2*in_len)]
    keep=np.any(w!=0,axis=0)
    return w[:,keep],idx[:,keep]

def matlab_bicubic_down(x,k):
    """Antialiased MATLAB-compatible bicubic downsampling by an integer factor.

    This is the degradation used to train standard bicubic (BI) super-resolution
    checkpoints, so pretrained baselines are evaluated in their native regime.
    """
    x=np.asarray(x,np.float64); h,w=x.shape[:2]
    wy,iy=_imresize_weights(h,h//k,1./k); wx,ix=_imresize_weights(w,w//k,1./k)
    y=np.einsum('ot,otwc->owc',wy,x[iy]) if x.ndim==3 else np.einsum('ot,otw->ow',wy,x[iy])
    y=np.einsum('ot,hotc->hoc',wx,y[:,ix]) if x.ndim==3 else np.einsum('ot,hot->ho',wx,y[:,ix])
    return y.astype(np.float32)

def up(x,k):return x.repeat(k,0).repeat(k,1)

def base_image(p,k):
    b=cv2.resize(p,(p.shape[1]*k,p.shape[0]*k),interpolation=cv2.INTER_CUBIC)
    return (b+up(p-down(b,k),k)).astype(np.float32)

def detail(x,k):return x-base_image(down(x,k),k)

def blocks(x,k):
    if x.ndim==3:x=x.mean(2)
    h,w=x.shape
    return x.reshape(h//k,k,w//k,k).mean((1,3))

def expand(x,k):return x.repeat(k,0).repeat(k,1)

def remap(x,flow):
    yy,xx=np.indices(flow.shape[:2],dtype=np.float32)
    return cv2.remap(x,xx+flow[...,0],yy+flow[...,1],cv2.INTER_LINEAR,borderMode=cv2.BORDER_CONSTANT)

def gray(x):return cv2.cvtColor(x.astype(np.float32),cv2.COLOR_RGB2GRAY)

def gradient_energy(x):
    y=gray(x) if x.ndim==3 else x
    gx=cv2.Sobel(y,cv2.CV_32F,1,0,ksize=3)/8
    gy=cv2.Sobel(y,cv2.CV_32F,0,1,ksize=3)/8
    return gx*gx+gy*gy

class RiskMemory:
    """Weighted noisy-observation risk minimisation; no clean pixel target."""
    def __init__(self,family='extra',use_history=True,leaf=40):
        self.family=family;self.use_history=use_history;self.leaf=leaf
    def _x(self,X):return X if self.use_history else X[:,:14]
    def fit(self,X,C,H,R):
        eps=1e-6;weights=(H+eps)/(H+eps).mean();X=self._x(X)
        if self.family=='extra':
            self.gain_model=ExtraTreesRegressor(n_estimators=80,min_samples_leaf=self.leaf,max_depth=16,n_jobs=1,random_state=42)
            self.risk_model=ExtraTreesRegressor(n_estimators=80,min_samples_leaf=self.leaf,max_depth=16,n_jobs=1,random_state=43)
        else:
            args=dict(max_iter=120,max_leaf_nodes=15,min_samples_leaf=self.leaf,l2_regularization=1.,learning_rate=.06,early_stopping=False)
            self.gain_model=HistGradientBoostingRegressor(random_state=42,**args)
            self.risk_model=HistGradientBoostingRegressor(loss='poisson',random_state=43,**args)
        self.gain_model.fit(X,C/(H+eps),sample_weight=weights)
        self.risk_model.fit(X,np.maximum(R,1e-8));return self
    def predict(self,X,H):
        raw=self.gain_model.predict(self._x(X));g=np.clip(raw,0,1)
        R=np.maximum(self.risk_model.predict(self._x(X)),0)
        return g,np.maximum(R-2*g*raw*(H+1e-6)+g*g*H,0)

def feature_maps(bank,age,anchor,history,obs,cfg):
    k=cfg.block;b=obs['b'];h=detail(bank,cfg.factor)
    H=blocks(h*h,k);e=blocks(obs['innovation'],k)
    a=blocks(age,k);grad=blocks(gradient_energy(b),k)
    old=blocks(gradient_energy(b+h),k)
    curv=blocks(cv2.Laplacian(gray(b),cv2.CV_32F)**2,k)
    ae=blocks((b-anchor)**2,k)
    hist=blocks(history,k)
    v=blocks(obs['fb'],k);speed=blocks(np.sum(obs['flow']**2,2),k)
    noise=obs['noise']**2; eps=1e-6
    log=lambda x:np.log1p(np.maximum(x,0)/eps)
    arrays=[log(H),log(e),np.minimum(e/(grad+noise+eps),100),np.log1p(a),np.minimum(a,64)/64,
        np.log1p(v),np.log1p(speed),log(grad),log(curv),np.full_like(H,log(noise)),log(old),
        np.minimum(H/(grad+noise+eps),100),log(e),log(blocks((b-cv2.GaussianBlur(b,(0,0),2.))**2,k)),
        log(ae),np.minimum(ae/(grad+noise+eps),100),log(hist)]
    X=np.stack(arrays,-1).reshape(-1,len(arrays)).astype(np.float32)
    return X,h,H,a,e,grad,ae,v

'''
(SOURCE_DIR / 'video_core.py').write_text(SOURCE, encoding='utf-8')
print('Wrote', 'video_core.py')

## 3. Audit estimators and gaze allocation

In [ ]:
SOURCE = r'''"""Audit estimators and gaze constraints."""
from dataclasses import dataclass
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import GroupKFold


@dataclass(frozen=True)
class MethodConfig:
    epsilon: float = .05
    kappa: float = 2.
    write_kappa: float = 2.0  # simultaneous bound for one-step write-value audit
    learning_rate: float = .15
    envelope_reserve: float = .15  # retained only for the supplied R7 reserve baseline
    certificate_kappa: float = 1.5  # >=1; hard observable-envelope importance cap
    online_moment_mix: float = .25
    moment_rate: float = .05
    alpha: float = .05
    cs_rho: float = .01
    sigma_bound: float = .10
    seed: int = 20260922

    def __post_init__(self):
        assert 0 < self.epsilon < 1 and self.kappa >= 1 and self.write_kappa >= 1
        assert 0 <= self.envelope_reserve < 1 and self.certificate_kappa >= 1
        assert 0 <= self.online_moment_mix <= 1
        assert self.learning_rate >= 0 and 0 < self.alpha < 1 and self.cs_rho > 0
        assert self.sigma_bound >= 0 and 0 < self.moment_rate <= 1


def project_simplex(x):
    """Euclidean projection onto nonnegative weights summing to one."""
    x = np.asarray(x, dtype=float)
    u = np.sort(x)[::-1]
    c = np.cumsum(u) - 1
    good = np.flatnonzero(u - c / np.arange(1, len(x) + 1) > 0)
    theta = c[good[-1]] / (good[-1] + 1)
    w = np.maximum(x - theta, 0)
    return w / w.sum()


def lower_normalize(weights, lower):
    """Water filling with arbitrary, feasible probability lower bounds."""
    w = np.maximum(np.asarray(weights, float), 1e-100)
    lower = np.broadcast_to(np.asarray(lower, float), w.shape).copy()
    if np.any(lower <= 0) or lower.sum() > 1 + 1e-12:
        raise ValueError('Infeasible probability lower bounds')
    if lower.sum() >= 1 - 1e-14:
        return lower / lower.sum()
    fixed = np.zeros(len(w), bool)
    for _ in range(len(w) + 1):
        p = lower.copy()
        free = ~fixed
        p[free] = (1 - lower[fixed].sum()) * w[free] / w[free].sum()
        bad = free & (p < lower - 1e-15)
        if not bad.any():
            return p / p.sum()
        fixed |= bad
    raise RuntimeError('Water filling failed')


def constrained_gaze(benefit, moment, lower, kappa=2.):
    """Maximize r.q subject to sum(v/q) <= kappa * sum(v/q_ref).

    q_ref = lower + (1-sum(lower))/J is always feasible. At constant lower
    bounds this recovers the R6 uniform-reference constraint. KKT bisection
    handles unequal envelope floors; near-degenerate cases return q_ref.
    """
    if kappa < 1:
        raise ValueError('kappa must be at least one')
    r = np.asarray(benefit, float)
    v = np.maximum(np.asarray(moment, float), 1e-16)
    l = np.broadcast_to(np.asarray(lower, float), r.shape).copy()
    if np.any(l <= 0) or l.sum() > 1 + 1e-12:
        raise ValueError('lower must be positive and sum to at most one')
    if l.sum() >= 1 - 1e-12:
        return l / l.sum()
    ref = l + (1 - l.sum()) / len(l)
    v = v / max(v.mean(), 1e-30)
    cap = float(kappa * np.sum(v / ref))
    moment_fn = lambda p: float(np.sum(v / p))
    greedy = l.copy()
    greedy[int(np.argmax(r))] += 1 - l.sum()
    if moment_fn(greedy) <= cap * (1 + 1e-12):
        return greedy
    if np.ptp(r) < 1e-14:
        return lower_normalize(np.sqrt(v), l)
    r = (r - r.min()) / np.ptp(r)
    def dist(nu):
        return lower_normalize(np.sqrt(v / np.maximum(nu - r, 1e-15)), l)
    lo, hi = 1., 2.
    for _ in range(64):
        if moment_fn(dist(hi)) <= cap:
            break
        hi = 1 + 2 * (hi - 1)
    else:
        return ref
    for _ in range(50):
        mid = (lo + hi) / 2
        if moment_fn(dist(mid)) > cap:
            lo = mid
        else:
            hi = mid
    p = dist(hi)
    # Numerical fallback is explicit and preserves feasibility.
    return p if moment_fn(p) <= cap * (1 + 1e-9) else ref


def contrast_bounds(base, prediction, tile):
    """Tight pixelwise extrema of d=2*x*(m-b)+b^2-m^2 for x in [0,1]."""
    from video_core import blocks
    slope = 2 * (prediction - base)
    intercept = base * base - prediction * prediction
    lo = blocks(intercept + np.minimum(slope, 0), tile).ravel().astype(float)
    hi = blocks(intercept + np.maximum(slope, 0), tile).ravel().astype(float)
    energy = blocks((prediction - base) ** 2, tile).ravel().astype(float)
    return lo, hi, energy


def envelope_lower(mu, lo, hi, noise_var_bound, cfg):
    """Reserve a fixed mass using observable residual and noise envelopes."""
    c = np.maximum(abs(lo - mu), abs(hi - mu)) + np.sqrt(noise_var_bound)
    mass = c / c.sum() if c.sum() > 1e-15 else np.full(len(c), 1 / len(c))
    return cfg.epsilon / len(c) + (1 - cfg.epsilon) * cfg.envelope_reserve * mass


def certificate_lower(mu, lo, hi, noise_var_bound, cfg):
    """Hard predictable probability certificate.

    Let c_j bound |d_j-mu_j| plus the declared Gaussian noise scale.  A
    distribution q_safe proportional to c (with the exploration floor) defines
    B_ref=max_j c_j/(J q_safe,j).  Requiring q_j >= c_j/(J*Gamma), with
    Gamma=certificate_kappa*B_ref, guarantees max_j c_j/(J q_j) <= Gamma.
    The guarantee is observable before the action and does not use learned
    residual calibration.  certificate_kappa=1 is the tightest reference cap.
    """
    c = np.maximum(abs(lo - mu), abs(hi - mu)) + np.sqrt(np.maximum(noise_var_bound, 0))
    j = len(c)
    floor = np.full(j, cfg.epsilon / j)
    q_safe = lower_normalize(np.maximum(c, 1e-15), floor)
    b_ref = float(np.max(c / np.maximum(j * q_safe, 1e-30)))
    gamma = max(cfg.certificate_kappa * b_ref, 1e-15)
    lower = np.maximum(floor, c / (j * gamma))
    # q_safe is feasible by construction; absorb only floating-point excess.
    if lower.sum() > 1 + 1e-10:
        raise RuntimeError('Certificate lower bounds became infeasible')
    if lower.sum() > 1:
        lower /= lower.sum()
    return lower, b_ref, gamma


def subgaussian_proxy(q, mu, lo, hi, noise_var_bound):
    """Hoeffding range proxy plus maximum conditional Gaussian variance.

    Gaussian noise must be independent across RGB scalar samples with SD
    no greater than the declared bound. The audit observation is not clipped.
    """
    j = len(q)
    left = mu.mean() + (lo - mu) / (j * q)
    right = mu.mean() + (hi - mu) / (j * q)
    width = float(right.max() - left.min())
    return width * width / 4 + float(np.max(noise_var_bound / (j * q) ** 2))


class NormalMixtureCS:
    def __init__(self, alpha=.05, rho=.01):
        self.alpha, self.rho = alpha, rho
        self.total = self.variance_proxy = 0.
        self.n = 0

    def update(self, estimate, predictable_proxy):
        self.total += float(estimate)
        self.variance_proxy += max(float(predictable_proxy), 0)
        self.n += 1
        v = self.variance_proxy + self.rho
        radius = np.sqrt(v * np.log(v / (self.rho * self.alpha ** 2))) / self.n
        center = self.total / self.n
        # Intersect with the known parameter domain; an empty interval signals
        # a coverage failure and is not silently repaired into a nonempty set.
        return max(-1., center - radius), min(1., center + radius), center


def audit_gradient(experts, weights, q, action, observed):
    """Unbiased stochastic gradient of the conditional design variance at fixed q.

    E[g | pre-action state] = -2/J^2 sum M_j e_j/q_j
                              +2*Mbar*(dbar-mubar).
    Taking a gradient through q would be a different objective; we do not do so.
    """
    j = len(q)
    mu = experts @ weights
    a = action
    return (-2 * experts[a] * (observed - mu[a]) / (j * q[a]) ** 2
            + 2 * experts.mean(0) * (observed / (j * q[a]) - mu.mean()))


class PredictableMixture:
    def __init__(self, cfg):
        self.cfg = cfg
        self.weights = np.array([0., 0., 1.])  # zero, energy, learned
        self.gradient_squared = 1e-12
        self.recent = None

    def moment(self, covariances):
        c = covariances
        if self.recent is not None:
            rate = self.cfg.online_moment_mix
            c = (1 - rate) * c + rate * self.recent[None]
        return np.maximum(np.einsum('k,jkl,l->j', self.weights, c, self.weights), 1e-14)

    def update(self, experts, q, action, observed, adapt_weights=True):
        # Called strictly AFTER storing the current audit and current q.
        if adapt_weights:
            g = audit_gradient(experts, self.weights, q, action, observed)
            self.gradient_squared += float(g @ g)
            step = self.cfg.learning_rate / np.sqrt(self.gradient_squared)
            self.weights = project_simplex(self.weights - step * g)
        residual = observed - experts[action]
        sample = np.outer(residual, residual) / (len(q) * q[action])
        rate = self.cfg.moment_rate
        self.recent = sample if self.recent is None else (1 - rate) * self.recent + rate * sample


class AuditEnsemble:
    """Source-cross-fitted three-expert residual second-moment matrices."""
    def __init__(self, trees=64, leaf=12):
        self.trees, self.leaf = trees, leaf

    def _regressor(self, seed):
        return ExtraTreesRegressor(n_estimators=self.trees, min_samples_leaf=self.leaf,
                                   max_depth=16, n_jobs=1, random_state=seed)

    def fit(self, features, observed, energy, groups):
        if len(np.unique(groups)) < 2:
            raise ValueError('Auditor needs at least two independent training groups')
        oof = np.empty(len(observed), float)
        cv = GroupKFold(n_splits=min(5, len(np.unique(groups))))
        for train, valid in cv.split(features, observed, groups):
            oof[valid] = self._regressor(104).fit(features[train], observed[train]).predict(features[valid])
        residual = observed[:, None] - np.column_stack([np.zeros(len(oof)), energy, np.clip(oof, -1, 1)])
        covariance = np.einsum('nk,nl->nkl', residual, residual)
        self.mean = self._regressor(104).fit(features, observed)
        self.second = self._regressor(105).fit(features, covariance.reshape(-1, 9))
        self.floor = max(float(np.quantile(residual[:, 2] ** 2, .10)), 1e-14)
        self.oof_mse = float(np.mean(residual[:, 2] ** 2))
        return self

    def predict(self, features, energy):
        mean = np.clip(self.mean.predict(features), -1, 1)
        experts = np.column_stack([np.zeros(len(mean)), np.clip(energy, 0, 1), mean])
        c = self.second.predict(features).reshape(-1, 3, 3)
        # A common multi-output tree partition averages PSD outer products.
        # Symmetrization and a diagonal ridge absorb floating point noise.
        c = (c + c.transpose(0, 2, 1)) / 2 + self.floor * np.eye(3)[None]
        return experts, c



def dual_audit_ray_gaze(benefit, primary_moment, write_moment, lower, kappa=2., write_kappa=2.):
    """Fast simultaneous two-audit allocation with explicit guarantees.

    The benefit-greedy distribution q0 and the feasible reference q_ref share
    identical positive lower bounds.  We search only the line segment
        q(alpha)=(1-alpha)q0 + alpha q_ref.
    Because each second-moment functional sum_j v_j/q_j is convex in q and
    q_ref is feasible, the feasible alpha values form an interval containing 1.
    Bisection therefore returns the smallest feasible alpha.  Since expected
    reconstruction benefit is linear in q and q0 maximizes it on this ray, this
    is the highest-benefit point on the ray satisfying *both* audit budgets.

    This is intentionally not claimed to solve the globally optimal
    multi-constraint simplex problem; it is a cheap, certifiable controller for
    the very large video benchmark.
    """
    if kappa < 1 or write_kappa < 1:
        raise ValueError('audit kappas must be at least one')
    r=np.asarray(benefit,float); v1=np.maximum(np.asarray(primary_moment,float),1e-16)
    v2=np.maximum(np.asarray(write_moment,float),0.)
    l=np.broadcast_to(np.asarray(lower,float),r.shape).copy()
    if np.any(l<=0) or l.sum()>1+1e-12: raise ValueError('infeasible lower bounds')
    if l.sum()>=1-1e-12:
        q=l/l.sum(); return q,1.0,1.0,1.0
    ref=l+(1-l.sum())/len(l)
    greedy=l.copy(); greedy[int(np.argmax(r))]+=1-l.sum()
    c1_ref=float(np.sum(v1/ref)); cap1=kappa*c1_ref
    active2=float(v2.sum())>1e-18
    c2_ref=float(np.sum(v2/ref)) if active2 else 1.0
    cap2=write_kappa*c2_ref
    def stats(q):
        a=float(np.sum(v1/q))/max(c1_ref,1e-30)
        b=float(np.sum(v2/q))/max(c2_ref,1e-30) if active2 else 0.0
        return a,b
    def feasible(q):
        a,b=stats(q); return a<=kappa*(1+1e-11) and (not active2 or b<=write_kappa*(1+1e-11))
    if feasible(greedy):
        a,b=stats(greedy); return greedy,0.0,a,b
    lo,hi=0.,1.
    for _ in range(60):
        mid=(lo+hi)/2; q=(1-mid)*greedy+mid*ref
        if feasible(q): hi=mid
        else: lo=mid
    q=(1-hi)*greedy+hi*ref
    a,b=stats(q)
    if not feasible(q):
        q=ref; hi=1.; a,b=stats(q)
    return q,float(hi),float(a),float(b)


def policy_distribution(policy, benefit, risk, moment, experts, lower, cfg, write_moment=None):
    n = len(benefit)
    const = np.full(n, cfg.epsilon / n)
    if policy == 'uniform':
        return np.full(n, 1 / n)
    if policy == 'greedy':
        p = const.copy(); p[int(np.argmax(benefit))] += 1 - cfg.epsilon
        return p
    if policy == 'risk_only':
        # True uncertainty/risk baseline; the supplied R7 notebook accidentally
        # reused reconstruction benefit here, making this comparator mislabeled.
        return lower_normalize(np.sqrt(np.maximum(risk, 1e-14)), const)
    if policy == 'softmax':
        z = (benefit - np.min(benefit)) / max(float(np.ptp(benefit)), 1e-12)
        z = np.exp((z - 1) / .25)
        return const + (1 - cfg.epsilon) * z / z.sum()
    if policy == 'audit_only':
        return lower_normalize(np.sqrt(moment), const)
    if policy == 'ppat_2026_m1':
        # Native M=1 centered-proxy PPAT proposal with learned residual
        # variance and surrogate conditional mean equal to the learned proxy.
        score = np.sqrt(moment + float(experts[:, 2].mean()) ** 2)
        return const + (1 - cfg.epsilon) * score / score.sum()
    if policy == 'joint':
        return constrained_gaze(benefit, moment, lower, cfg.kappa)
    if policy == 'dual_joint':
        if write_moment is None:
            raise ValueError('dual_joint requires write_moment')
        return dual_audit_ray_gaze(benefit, moment, write_moment, lower, cfg.kappa, cfg.write_kappa)[0]
    raise ValueError(policy)


def exact_variance(d, mu, q, noise_var):
    e = d - mu
    return max(float(np.sum((e * e + noise_var) / q) / len(q) ** 2 - e.mean() ** 2), 0.)


def variance_quadratic(experts, d, q, noise_var):
    """Return A,b,c for V(w)=w'A w+2 b'w+c at a fixed realized state.

    This is evaluator-only and is used after the trajectory to measure online
    mixture regret against the best *fixed* convex expert mixture in hindsight.
    It never changes the executed actions or estimates.
    """
    M = np.asarray(experts, float)
    d = np.asarray(d, float)
    q = np.asarray(q, float)
    nv = np.asarray(noise_var, float)
    j = len(q)
    mbar = M.mean(0)
    dbar = float(d.mean())
    A = M.T @ (M / q[:, None]) / (j*j) - np.outer(mbar, mbar)
    b = -(M.T @ (d / q)) / (j*j) + dbar * mbar
    c = float(np.sum((d*d + nv) / q) / (j*j) - dbar*dbar)
    A = (A + A.T) / 2
    return A, b, c


def best_fixed_mixture(A, b, c):
    """Convex 3-expert hindsight oracle; diagnostic, not a deployable method."""
    from scipy.optimize import minimize
    A = (np.asarray(A, float) + np.asarray(A, float).T) / 2
    b = np.asarray(b, float)
    fun = lambda w: float(w @ A @ w + 2*b @ w + c)
    jac = lambda w: 2*A @ w + 2*b
    res = minimize(fun, np.full(3, 1/3), jac=jac, method='SLSQP',
                   bounds=[(0.,1.)]*3,
                   constraints={'type':'eq','fun':lambda w: float(w.sum()-1),
                                'jac':lambda w: np.ones(3)},
                   options={'ftol':1e-12,'maxiter':500})
    candidates = [np.eye(3)[k] for k in range(3)]
    if res.success and np.isfinite(res.fun):
        candidates.append(project_simplex(res.x))
    vals = [fun(w) for w in candidates]
    k = int(np.argmin(vals))
    return candidates[k], max(float(vals[k]), 0.)


def fixed_pool_ppat(loss, proxy, scores, sample_count, rng, epsilon=.05):
    """Native finite-pool LURE/PPAT, fixed lambda=1, without replacement.

    Diagnostic only: freeze a state and do not update memory during this assay.
    This extra dense-reference experiment is not a one-patch sensor result.
    """
    n, m = len(loss), sample_count
    if not 1 <= m <= n:
        raise ValueError('Invalid query budget')
    remaining = list(range(n)); vals, residuals = [], []
    centered = proxy - proxy.mean()
    for k in range(1, m + 1):
        score = np.maximum(scores[remaining], 1e-15)
        q = epsilon / len(remaining) + (1 - epsilon) * score / score.sum()
        pos = int(rng.choice(len(remaining), p=q)); a = remaining.pop(pos)
        weight = 1. if m == n else 1 + (n - m) / (n - k) * (1 / ((n - k + 1) * q[pos]) - 1)
        vals.append(weight * loss[a]); residuals.append(weight * (loss[a] - centered[a]))
    return float(np.mean(vals)), float(np.mean(residuals))
'''
(SOURCE_DIR / 'r9_core.py').write_text(SOURCE, encoding='utf-8')
print('Wrote', 'r9_core.py')

## 4. Data loading and integrity

In [ ]:
SOURCE = r'''"""Dataset loading and checks."""
from pathlib import Path
from dataclasses import asdict, dataclass
import hashlib, json, os, re, time, urllib.request, urllib.error, zipfile
import cv2
import numpy as np

DATASETS = {
    'DAVIS': {'url': 'https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip',
              'sha256': 'e3d0b5b77c3d031b000a19e0e25e3e2cac65d183755601bc2cf066df1a2aa492',
              'source': 'https://davischallenge.org/davis2017/code.html'},
    'REDS': {'url': 'https://huggingface.co/datasets/snah/REDS/resolve/main/val_sharp.zip',
             'source': 'https://seungjunnah.github.io/Datasets/reds.html',
             'split': 'val_sharp, 30 clips x 100 frames'},
    'REDS_train': {'url': None,
                   'source': 'https://seungjunnah.github.io/Datasets/reds.html',
                   'split': 'train_sharp, 240 clips x 100 frames',
                   'note': 'Supplied as a local copy. Replaces Vimeo-90K (official download unavailable). '
                           'No model in this study is trained on REDS, so it is unseen external test data.'},
}


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for part in iter(lambda: f.read(2 ** 20), b''):
            h.update(part)
    return h.hexdigest()


def digest_json(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, separators=(',', ':'), default=str).encode()).hexdigest()


def atomic_json(path, obj):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(p.name + '.partial')
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True, default=lambda x: x.item() if hasattr(x, 'item') else str(x)), encoding='utf-8')
    os.replace(tmp, p)


def immutable_json(path, obj):
    p = Path(path)
    if p.exists():
        if json.loads(p.read_text()) != json.loads(json.dumps(obj, default=str)):
            raise RuntimeError(f'Locked inputs changed: {p}. Choose a new experiment directory.')
    else:
        atomic_json(p, obj)
    return obj


def download(url, path, expected=None):
    """Download with resume support."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    record = path.with_name(path.name + '.download.json')
    if path.exists():
        observed = sha256(path)
        old = json.loads(record.read_text()) if record.exists() else {}
        check = expected or old.get('sha256')
        if check and observed != check:
            raise ValueError(f'Content changed or corrupt: {path}')
        if not record.exists():
            atomic_json(record, {'url': url, 'sha256': observed, 'bytes': path.stat().st_size,
                                 'verification': 'upstream-pinned' if expected else 'first-observed'})
        return path
    partial = path.with_name(path.name + '.partial')
    for attempt in range(3):
        try:
            size = partial.stat().st_size if partial.exists() else 0
            headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124 Safari/537.36',
                       'Accept': '*/*', 'Referer': 'https://toflow.csail.mit.edu/'}
            if size:
                headers['Range'] = f'bytes={size}-'
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=60) as r:
                resumed = size > 0 and r.status == 206
                if resumed and not r.headers.get('Content-Range', '').startswith(f'bytes {size}-'):
                    raise RuntimeError('Invalid server resume offset')
                total = size if resumed else 0; announced = total
                with partial.open('ab' if resumed else 'wb') as f:
                    while True:
                        part = r.read(2 ** 20)
                        if not part:
                            break
                        f.write(part); total += len(part)
                        if total - announced >= 128 * 2 ** 20:
                            print(path.name, round(total / 2 ** 30, 2), 'GiB downloaded', flush=True)
                            announced = total
            h = sha256(partial)
            if expected and h != expected:
                partial.unlink(missing_ok=True)
                raise ValueError(f'Checksum mismatch: {path.name}')
            os.replace(partial, path)
            atomic_json(record, {'url': url, 'sha256': h, 'bytes': path.stat().st_size,
                                 'verification': 'upstream-pinned' if expected else 'first-observed'})
            return path
        except Exception:
            if attempt == 2:
                raise


def extract_dataset(archive, destination, dataset):
    destination = Path(destination); destination.mkdir(parents=True, exist_ok=True)
    stamp = destination / 'extraction.json'
    fingerprint = {'archive_sha256': sha256(archive), 'dataset': dataset}
    if stamp.exists() and json.loads(stamp.read_text()) == fingerprint:
        return destination
    with zipfile.ZipFile(archive) as z:
        for entry in z.infolist():
            rel = Path(entry.filename)
            if rel.is_absolute() or '..' in rel.parts:
                raise ValueError('Unsafe archive member')
            if entry.is_dir():
                continue
            name = entry.filename
            if dataset == 'DAVIS':
                keep = '/JPEGImages/480p/' in name or '/ImageSets/2017/' in name
            else:
                keep = name.lower().endswith(('.png', '.jpg'))
            if keep:
                p = destination / rel; p.parent.mkdir(parents=True, exist_ok=True)
                # Opening and reading a member verifies its CRC, including after resume.
                with z.open(entry) as src, p.open('wb') as dst:
                    while True:
                        b = src.read(2 ** 20)
                        if not b: break
                        dst.write(b)
    atomic_json(stamp, fingerprint)
    return destination


@dataclass(frozen=True)
class Clip:
    dataset: str
    clip_id: str
    source_group: str
    role: str
    paths: tuple
    content_sha256: str


def _clip(dataset, name, group, role, files):
    paths = tuple(str(p.resolve()) for p in files)
    h = hashlib.sha256()
    for p in files:
        h.update(sha256(p).encode())
    return Clip(dataset, name, f'{dataset}:{group}', role, paths, h.hexdigest())


def discover_davis(root, split, frames):
    roots = list(Path(root).rglob('JPEGImages'))
    if len(roots) != 1:
        raise ValueError(f'Expected one DAVIS JPEGImages directory in {root}')
    p = roots[0] / '480p'; clips = []
    role_map = {'fit': 'fit', 'development': 'development', 'reserved': 'legacy_test', 'test': 'legacy_test'}
    for key, names in split.items():
        for name in names:
            files = sorted((p / name).glob('*.jpg'))[:frames]
            if len(files) < 7:
                raise ValueError(f'Missing DAVIS clip: {name}')
            clips.append(_clip('DAVIS', name, name, role_map[key], files))
    return clips


def discover_reds(root, frames=100, limit=None, dataset='REDS', expected=None):
    """One REDS split folder (val_sharp or train_sharp) -> one clip per subfolder.

    Train and val clip folders both start at 000, so each split is registered as
    its own dataset ('REDS' for val_sharp, 'REDS_train' for train_sharp); clip IDs
    and source groups therefore never collide. `expected` is the number of clips
    the split must contain; `limit` keeps the first clips by name (pilot runs).
    """
    parents = sorted({p.parent for p in Path(root).rglob('*.png')}, key=lambda p: p.name)
    if not parents:
        raise ValueError(f'{dataset}: no PNG frames found in {root}')
    if len({p.name for p in parents}) != len(parents):
        raise ValueError(f'Ambiguous {dataset} directories in {root}; point to one split folder (val_sharp or train_sharp) only')
    if expected is not None and len(parents) != expected:
        raise ValueError(f'Incomplete {dataset} split: found {len(parents)} / {expected} clips in {root}')
    clips = []
    for p in (parents if limit is None else parents[:limit]):
        files = sorted(p.glob('*.png'))[:frames]
        if len(files) < 7:
            raise ValueError(f'{dataset} clip {p.name} has fewer than 7 frames')
        clips.append(_clip(dataset, p.name, p.name, 'external_test', files))
    return clips


def validate_manifest(clips):
    if not clips:
        raise ValueError('Empty dataset manifest')
    keys = [(c.dataset, c.clip_id) for c in clips]
    assert len(set(keys)) == len(keys), 'Duplicate clip identity'
    roles, hashes = {}, {}
    for c in clips:
        roles.setdefault(c.source_group, set()).add(c.role)
        hashes.setdefault(c.content_sha256, []).append(c)
    for name, role in roles.items():
        assert len(role) == 1, f'Source crosses partitions: {name}'
    for vals in hashes.values():
        assert len({v.role for v in vals}) == 1, 'Identical content crosses partitions'
    duplicates = [[v.clip_id for v in vals] for vals in hashes.values() if len(vals) > 1]
    # Exact duplicates within evaluation get one canonical source group so
    # inference never counts their duplicated clips as independent sources.
    from dataclasses import replace
    parent={c.source_group:c.source_group for c in clips}
    def find(g):
        while parent[g]!=g:
            parent[g]=parent[parent[g]];g=parent[g]
        return g
    for vals in hashes.values():
        roots=sorted({find(c.source_group) for c in vals})
        for g in roots:parent[g]=roots[0]
    clips = [replace(c, source_group=find(c.source_group)) for c in clips]
    return clips, {'exact_duplicate_sets': duplicates, 'source_separation': True,
                   'reds_grouping': 'each REDS clip folder is one source cluster; REDS publishes no raw-video identity mapping, so near-duplicate scenes across clips cannot be excluded'}


def load_clip(clip, cfg):
    frames = []
    for path in clip.paths[:cfg.frames]:
        im = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if im is None:
            raise ValueError(f'Cannot decode image: {path}')
        im = im[..., ::-1]; h, w = im.shape[:2]
        if h < cfg.height or w < cfg.width:
            raise ValueError(f'{clip.clip_id}: source {h}x{w} smaller than requested crop {cfg.height}x{cfg.width}')
        y, x = (h - cfg.height) // 2, (w - cfg.width) // 2
        frames.append(im[y:y+cfg.height, x:x+cfg.width].astype(np.float32) / 255.)
    return np.stack(frames)


def synthetic_frames(cfg, seed=1, cut=False):
    """A software test fixture, never a natural-video benchmark substitute."""
    rng = np.random.default_rng(seed)
    texture = rng.uniform(0, 1, (cfg.height, cfg.width, 3)).astype(np.float32)
    texture = cv2.GaussianBlur(texture, (0, 0), .5)
    out = []
    for t in range(cfg.frames):
        im = np.roll(texture, (t, 2*t), axis=(0, 1)).copy()
        if cut and t >= cfg.frames // 2:
            im = 1 - im
        out.append(im)
    return np.stack(out)
'''
(SOURCE_DIR / 'r9_data.py').write_text(SOURCE, encoding='utf-8')
print('Wrote', 'r9_data.py')

## 5. Published SR and video-SR baselines

In [ ]:
SOURCE = r'''"""Recent SR baselines."""
from pathlib import Path
import importlib.util
import hashlib
import time
import numpy as np
from r9_data import download, atomic_json, sha256

COMMIT = 'b69fb668c0362deb696eecbfdebfaa5c2fcdfcb4'
BASE = f'https://raw.githubusercontent.com/Amazingren/NTIRE2025_ESR/{COMMIT}/'
SPECS = {
 'NanoSR2025': dict(code='models/team07_NanoSR.py', weights='model_zoo/team07_NanoSR.pth',
                   code_sha='a001a75884786651849f1561de6ba0767ecf00cce43c20bc621de9b428a5d87e',
                   weights_sha='7c584c8ad7c79ee2b20e923ed610f0b40992ac66e7e8edafd19b45c9a0ba3e49',
                   cls='NanoSR_inference', args=[3, 3], kwargs={}, state=None),
 'SPANF2025': dict(code='models/team24_SPANF.py', weights='model_zoo/team24_spanf.pth',
                  code_sha='f4ae70c75cc51cdba090c594dbd52525b61870d3ede36099e43d4a38bbe1fbf1',
                  weights_sha='1508c566008654646bea1edff537c83dd7a6fe8f6ffda3f25f9eb7ba09cbbcd3',
                  cls='SPANF', args=[3, 3], kwargs={'upscale': 4, 'feature_channels': 32}, state=None),
 'SCMSR2025': dict(code='models/team16_SCMSR.py', weights='model_zoo/team16_SCMSR.pth',
                  code_sha='cd78b85b1f83aafd41085a2a5c3f8dc2e25ecf24f1f3b95cce921ebc5b746764',
                  weights_sha='68a8f2c24743050bc510c813f065d3379671cb2bdd6ee83c0bcb15f14f817deb',
                  cls='SCMSR', args=[], kwargs={}, state='params_ema'),
}


class OfficialSR:
    def __init__(self, name, root, device='auto'):
        import torch
        self.torch = torch; self.name = name
        self.device = ('cuda' if torch.cuda.is_available() else 'cpu') if device == 'auto' else device
        spec = SPECS[name]; root = Path(root) / name; root.mkdir(parents=True, exist_ok=True)
        download(BASE+'LICENSE',root/'LICENSE','81032fbef39a967bdc9dc7f5aaaa3b65fe589c931c0cd38c473ecdf493b0e17a')
        source = download(BASE + spec['code'], root / Path(spec['code']).name, spec['code_sha'])
        weights = download(BASE + spec['weights'], root / Path(spec['weights']).name, spec['weights_sha'])
        code = source.read_text(encoding='utf-8'); patch = 'none'
        if name == 'SPANF2025':
            line = '        self.cuda()(torch.randn(1, 3, 256, 256).cuda())'
            if code.count(line) != 1:
                raise RuntimeError('Official SPAN-F warm-up changed; review the pinned source')
            code = code.replace(line, '        # Constructor-only CUDA warm-up omitted for CPU portability.')
            patch = 'remove_constructor_cuda_warmup_only'
        portable = root / 'portable_model.py'; portable.write_text(code, encoding='utf-8')
        module_spec = importlib.util.spec_from_file_location('official_' + name, portable)
        module = importlib.util.module_from_spec(module_spec); module_spec.loader.exec_module(module)
        model = getattr(module, spec['cls'])(*spec['args'], **spec['kwargs'])
        state = torch.load(weights, map_location='cpu', weights_only=True)
        if spec['state'] is not None:
            state = state[spec['state']]
        model.load_state_dict(state, strict=True)
        self.model = model.to(self.device).eval()
        self.parameters = int(sum(p.numel() for p in model.parameters()))
        self.provenance = dict(name=name, year=2025, repository='https://github.com/Amazingren/NTIRE2025_ESR',
            commit=COMMIT, source_sha256=sha256(source), portable_source_sha256=sha256(portable),
            checkpoint_sha256=sha256(weights), patch=patch, parameter_count=self.parameters,
            device=self.device, scale=4, precision='float32', input='RGB [0,1]',
            adaptation='current LR preview; optional identical causal residual memory and paid patch',
            pretrained_training_data='external author checkpoint; see NTIRE 2025 report; cross-dataset overlap not fully auditable')
        atomic_json(root / 'model_provenance.json', self.provenance)
        self.warmup()

    causal = True
    temporal = False

    def predict_sequence(self, previews):
        """Frame-by-frame single-image SR; trivially causal."""
        preds, times = [], []
        for p in previews:
            y, ms = self.predict(p)
            preds.append(y); times.append(ms)
        return preds, times

    def sync(self):
        if self.device.startswith('cuda'):
            self.torch.cuda.synchronize()

    def warmup(self):
        for _ in range(2):
            self.predict(np.full((32, 32, 3), .5, np.float32))

    def predict(self, preview):
        torch = self.torch
        self.sync(); wall_start = time.perf_counter()
        x = torch.from_numpy(np.ascontiguousarray(np.clip(preview, 0, 1).transpose(2, 0, 1))).unsqueeze(0).to(self.device)
        h, w = x.shape[-2:]
        # Preserve each author's native padding and input normalization.
        # SCMSR performs its own reflection padding inside forward_origin.
        self.sync(); start = time.perf_counter()
        with torch.inference_mode():
            out = self.model(x)
        self.sync(); ms = 1000 * (time.perf_counter() - start)
        if isinstance(out, (tuple, list)):
            raise TypeError('Unexpected official model output')
        y = out[0, :, :h*4, :w*4].detach().float().cpu().numpy().transpose(1, 2, 0)
        if y.shape != (h*4, w*4, 3) or not np.isfinite(y).all():
            raise ValueError('Invalid pretrained model prediction')
        # Charge conversion and device transfers as well as model execution.
        y = np.clip(y, 0, 1).astype(np.float32)
        return y, 1000*(time.perf_counter()-wall_start)


# ---------------------------------------------------------------------------
# Video super-resolution baseline: BasicVSR++ (CVPR 2022), official OpenMMLab
# Vimeo-90K BI checkpoint. It is trained on the Vimeo-90K *training* split only,
# so REDS (train_sharp and val_sharp) and DAVIS are unseen.
# (The REDS4 checkpoint is deliberately NOT used: it is trained on REDS train+val.)
# ---------------------------------------------------------------------------
VSR_SPEC = dict(
    name='BasicVSRpp2022',
    urls=['https://download.openmmlab.com/mmediting/restorers/basicvsr_plusplus/'
          'basicvsr_plusplus_c64n7_8x1_300k_vimeo90k_bi_20210305-4ef437e2.pth',
          'https://download.openmmlab.com/mmediting/restorers/basicvsr_plusplus/'
          'basicvsr_plusplus_c64n7_4x2_300k_vimeo90k_bi_20210305-4ef437e2.pth'],
    sha256_prefix='4ef437e2',
    paper='K. C. K. Chan et al., BasicVSR++, CVPR 2022',
    training_data='Vimeo-90K septuplet training split, bicubic x4 degradation')


def _load_vsr_state(path):
    import torch
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    state = ckpt.get('state_dict', ckpt)
    gen = {k[len('generator.'):]: v for k, v in state.items() if k.startswith('generator.')}
    if not gen:
        gen = {k: v for k, v in state.items() if not k.startswith('step_counter')}
    return gen


class TemporalWindowSR:
    """Shared-weight video SR wrapper.

    mode='causal': the prediction for frame t uses only previews max(0,t-W+1)..t,
    the same information available to every other online method.
    mode='offline': one bidirectional pass over the whole clip (reads FUTURE
    previews); reported only as a non-causal reference.
    """
    temporal = True

    def __init__(self, net, provenance, device, mode='causal', window=7):
        assert mode in ['causal', 'offline'] and window >= 2
        import torch
        self.torch = torch; self.net = net; self.device = device
        self.mode = mode; self.window = window; self.causal = mode == 'causal'
        self.name = provenance['name'] + ('_causal' if self.causal else '_offline')
        self.parameters = provenance['parameter_count']
        self.provenance = dict(provenance, name=self.name, mode=mode,
                               window=(window if self.causal else 'full clip'),
                               causal=self.causal)

    def sync(self):
        if str(self.device).startswith('cuda'):
            self.torch.cuda.synchronize()

    def _run(self, stack):
        """stack: (t,h,w,3) float32 in [0,1] -> (t,4h,4w,3)."""
        torch = self.torch
        t, h, w, _ = stack.shape
        ph, pw = max(64, -(-h // 4) * 4), max(64, -(-w // 4) * 4)
        x = torch.from_numpy(np.ascontiguousarray(np.clip(stack, 0, 1).transpose(0, 3, 1, 2))).float()
        x = x.unsqueeze(0).to(self.device)
        if (ph, pw) != (h, w):
            x = torch.nn.functional.pad(x.view(t, 3, h, w), (0, pw - w, 0, ph - h), mode='replicate').view(1, t, 3, ph, pw)
        if t == 1:
            x = torch.cat([x, x], dim=1)
        with torch.inference_mode():
            y = self.net(x)[0, :t, :, :h * 4, :w * 4]
        y = y.float().cpu().numpy().transpose(0, 2, 3, 1)
        if not np.isfinite(y).all():
            raise ValueError('Invalid video SR prediction')
        return np.clip(y, 0, 1).astype(np.float32)

    def predict_sequence(self, previews):
        stack = np.stack([np.asarray(p, np.float32) for p in previews])
        if not self.causal:
            self.sync(); start = time.perf_counter()
            out = self._run(stack); self.sync()
            ms = 1000 * (time.perf_counter() - start) / len(stack)
            return list(out), [ms] * len(stack)
        preds, times = [], []
        for t in range(len(stack)):
            self.sync(); start = time.perf_counter()
            window = stack[max(0, t - self.window + 1):t + 1]
            preds.append(self._run(window)[-1]); self.sync()
            times.append(1000 * (time.perf_counter() - start))
        return preds, times


def load_video_sr(root, device='auto', window=7):
    """Download (or reuse) the official checkpoint and build causal + offline wrappers."""
    import torch
    from r9_vsr_arch import BasicVSRPlusPlusNet
    device = ('cuda' if torch.cuda.is_available() else 'cpu') if device == 'auto' else device
    folder = Path(root) / VSR_SPEC['name']; folder.mkdir(parents=True, exist_ok=True)
    path = None; errors = []
    local = sorted(folder.glob('*.pth'))
    for p in local:
        if sha256(p).startswith(VSR_SPEC['sha256_prefix']):
            path = p; break
    if path is None:
        for url in VSR_SPEC['urls']:
            try:
                candidate = download(url, folder / url.rsplit('/', 1)[1])
            except Exception as error:
                errors.append(f'{url}: {type(error).__name__}: {error}'); continue
            if sha256(candidate).startswith(VSR_SPEC['sha256_prefix']):
                path = candidate; break
            errors.append(f'{url}: SHA-256 prefix mismatch'); candidate.unlink()
    if path is None:
        raise RuntimeError('BasicVSR++ Vimeo-90K BI checkpoint unavailable. Download it from the mmagic '
                           'BasicVSR++ model zoo and place the .pth file in ' + str(folder) + '. ' + ' | '.join(errors))
    net = BasicVSRPlusPlusNet(mid_channels=64, num_blocks=7)
    state = _load_vsr_state(path)
    missing, unexpected = net.load_state_dict(state, strict=False)
    missing = [k for k in missing if k not in ('spynet.mean', 'spynet.std')]
    if missing or unexpected:
        raise RuntimeError(f'BasicVSR++ checkpoint/architecture mismatch. Missing={missing[:8]} Unexpected={unexpected[:8]}')
    net = net.to(device).eval()
    provenance = dict(name=VSR_SPEC['name'], year=2022, paper=VSR_SPEC['paper'],
                      repository='https://github.com/open-mmlab/mmagic', checkpoint_file=path.name,
                      checkpoint_sha256=sha256(path), parameter_count=int(sum(p.numel() for p in net.parameters())),
                      training_data=VSR_SPEC['training_data'], device=device, scale=4, precision='float32',
                      implementation='mmagic-compatible PyTorch port with torchvision deform_conv2d; strict key check; behavioural sanity gate',
                      adaptation='current (and, for causal mode, past) LR previews; optional identical causal residual memory and paid patch')
    atomic_json(folder / 'model_provenance.json', provenance)
    causal = TemporalWindowSR(net, provenance, device, 'causal', window)
    offline = TemporalWindowSR(net, provenance, device, 'offline', window)
    causal.predict_sequence([np.full((64, 64, 3), .5, np.float32)] * 3)
    return {causal.name: causal, offline.name: offline}


def load_baselines(root, device='auto', include_video=True):
    models = {name: OfficialSR(name, root, device) for name in SPECS}
    if include_video:
        models.update(load_video_sr(root, device))
    return models


def baseline_sanity(models, clips_frames, factor=4):
    """Behavioural gate: each pretrained SR model must beat bicubic upsampling
    on clean, MATLAB-bicubic-downsampled held-out frames. A mis-ported or
    mis-loaded network fails loudly instead of producing a weak baseline."""
    import cv2
    from video_core import matlab_bicubic_down
    report = {}
    for name, model in models.items():
        gains = []
        for k, frames in enumerate(clips_frames):
            # Reproducible stochastic inference (SCMSR samples gumbel_softmax in eval mode).
            import torch
            from video_core import seed
            torch.manual_seed(seed('sanity', name, k))
            previews = [matlab_bicubic_down(f, factor) for f in frames]
            preds, _ = model.predict_sequence(previews)
            for f, p, y in zip(frames, previews, preds):
                up = np.clip(cv2.resize(p, (f.shape[1], f.shape[0]), interpolation=cv2.INTER_CUBIC), 0, 1)
                b = 8
                mse_sr = float(np.mean((y[b:-b, b:-b] - f[b:-b, b:-b]) ** 2))
                mse_bi = float(np.mean((up[b:-b, b:-b] - f[b:-b, b:-b]) ** 2))
                gains.append(10 * np.log10(mse_bi / max(mse_sr, 1e-12)))
        report[name] = dict(mean_psnr_gain_over_bicubic_db=float(np.mean(gains)), frames=len(gains),
                            passed=bool(np.mean(gains) > 0.0))
        if not report[name]['passed']:
            raise RuntimeError(f'Baseline sanity gate failed for {name}: {report[name]}')
    return report
'''
(SOURCE_DIR / 'r9_neural.py').write_text(SOURCE, encoding='utf-8')
print('Wrote', 'r9_neural.py')

## 6. BasicVSR++ architecture (official-checkpoint compatible)

In [ ]:
SOURCE = r'''"""BasicVSR++ (Chan et al., CVPR 2022) inference architecture.

Parameter names follow the official OpenMMLab (mmediting/mmagic) generator so the
official checkpoints load with strict key checking. Modulated deformable
convolution uses torchvision.ops.deform_conv2d, whose offset/mask layout matches
the mmcv DCNv2 kernel. Loading is followed by a behavioural sanity gate in the
runner (the network must beat bicubic upsampling on held-out clean frames), so an
incorrect port cannot silently enter the comparison.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.ops import deform_conv2d


def flow_warp(x, flow, interpolation='bilinear', padding_mode='zeros', align_corners=True):
    n, _, h, w = x.size()
    grid_y, grid_x = torch.meshgrid(torch.arange(0, h, device=x.device, dtype=x.dtype),
                                    torch.arange(0, w, device=x.device, dtype=x.dtype), indexing='ij')
    grid = torch.stack((grid_x, grid_y), 2)
    grid_flow = grid + flow
    gx = 2.0 * grid_flow[:, :, :, 0] / max(w - 1, 1) - 1.0
    gy = 2.0 * grid_flow[:, :, :, 1] / max(h - 1, 1) - 1.0
    return F.grid_sample(x, torch.stack((gx, gy), dim=3), mode=interpolation,
                         padding_mode=padding_mode, align_corners=align_corners)


class ResidualBlockNoBN(nn.Module):
    def __init__(self, mid_channels=64):
        super().__init__()
        self.conv1 = nn.Conv2d(mid_channels, mid_channels, 3, 1, 1, bias=True)
        self.conv2 = nn.Conv2d(mid_channels, mid_channels, 3, 1, 1, bias=True)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return x + self.conv2(self.relu(self.conv1(x)))


class ResidualBlocksWithInputConv(nn.Module):
    def __init__(self, in_channels, out_channels=64, num_blocks=30):
        super().__init__()
        main = [nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias=True), nn.LeakyReLU(0.1, inplace=True)]
        main.append(nn.Sequential(*[ResidualBlockNoBN(out_channels) for _ in range(num_blocks)]))
        self.main = nn.Sequential(*main)

    def forward(self, feat):
        return self.main(feat)


class PixelShufflePack(nn.Module):
    def __init__(self, in_channels, out_channels, scale_factor, upsample_kernel):
        super().__init__()
        self.scale_factor = scale_factor
        self.upsample_conv = nn.Conv2d(in_channels, out_channels * scale_factor * scale_factor,
                                       upsample_kernel, padding=(upsample_kernel - 1) // 2)

    def forward(self, x):
        return F.pixel_shuffle(self.upsample_conv(x), self.scale_factor)


class _ConvModule(nn.Module):
    def __init__(self, cin, cout, act=True):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, 7, 1, 3, bias=True)
        self.act = act

    def forward(self, x):
        x = self.conv(x)
        return F.relu(x) if self.act else x


class SPyNetBasicModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.basic_module = nn.Sequential(_ConvModule(8, 32), _ConvModule(32, 64), _ConvModule(64, 32),
                                          _ConvModule(32, 16), _ConvModule(16, 2, act=False))

    def forward(self, x):
        return self.basic_module(x)


class SPyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.basic_module = nn.ModuleList([SPyNetBasicModule() for _ in range(6)])
        self.register_buffer('mean', torch.Tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.Tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def compute_flow(self, ref, supp):
        n, _, h, w = ref.size()
        ref = [(ref - self.mean) / self.std]
        supp = [(supp - self.mean) / self.std]
        for _ in range(5):
            ref.append(F.avg_pool2d(ref[-1], 2, 2, count_include_pad=False))
            supp.append(F.avg_pool2d(supp[-1], 2, 2, count_include_pad=False))
        ref, supp = ref[::-1], supp[::-1]
        flow = ref[0].new_zeros(n, 2, h // 32, w // 32)
        for level in range(len(ref)):
            flow_up = flow if level == 0 else F.interpolate(flow, scale_factor=2, mode='bilinear', align_corners=True) * 2.0
            flow = flow_up + self.basic_module[level](torch.cat([
                ref[level], flow_warp(supp[level], flow_up.permute(0, 2, 3, 1), padding_mode='border'), flow_up], 1))
        return flow

    def forward(self, ref, supp):
        h, w = ref.shape[2:4]
        w_up = w if w % 32 == 0 else 32 * (w // 32 + 1)
        h_up = h if h % 32 == 0 else 32 * (h // 32 + 1)
        ref = F.interpolate(ref, size=(h_up, w_up), mode='bilinear', align_corners=False)
        supp = F.interpolate(supp, size=(h_up, w_up), mode='bilinear', align_corners=False)
        flow = F.interpolate(self.compute_flow(ref, supp), size=(h, w), mode='bilinear', align_corners=False)
        flow[:, 0] *= float(w) / float(w_up)
        flow[:, 1] *= float(h) / float(h_up)
        return flow


class SecondOrderDeformableAlignment(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1, deform_groups=16, max_residue_magnitude=10):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(out_channels, in_channels, kernel_size, kernel_size))
        self.bias = nn.Parameter(torch.zeros(out_channels))
        self.padding = padding
        self.deform_groups = deform_groups
        self.max_residue_magnitude = max_residue_magnitude
        self.conv_offset = nn.Sequential(
            nn.Conv2d(3 * out_channels + 4, out_channels, 3, 1, 1), nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1), nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1), nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(out_channels, 27 * deform_groups, 3, 1, 1))

    def forward(self, x, extra_feat, flow_1, flow_2):
        out = self.conv_offset(torch.cat([extra_feat, flow_1, flow_2], dim=1))
        o1, o2, mask = torch.chunk(out, 3, dim=1)
        offset = self.max_residue_magnitude * torch.tanh(torch.cat((o1, o2), dim=1))
        offset_1, offset_2 = torch.chunk(offset, 2, dim=1)
        offset_1 = offset_1 + flow_1.flip(1).repeat(1, offset_1.size(1) // 2, 1, 1)
        offset_2 = offset_2 + flow_2.flip(1).repeat(1, offset_2.size(1) // 2, 1, 1)
        offset = torch.cat([offset_1, offset_2], dim=1)
        return deform_conv2d(x, offset, self.weight, self.bias, stride=1, padding=self.padding,
                             dilation=1, mask=torch.sigmoid(mask))


class BasicVSRPlusPlusNet(nn.Module):
    def __init__(self, mid_channels=64, num_blocks=7, max_residue_magnitude=10):
        super().__init__()
        self.mid_channels = mid_channels
        self.spynet = SPyNet()
        self.feat_extract = ResidualBlocksWithInputConv(3, mid_channels, 5)
        self.deform_align = nn.ModuleDict()
        self.backbone = nn.ModuleDict()
        for i, module in enumerate(['backward_1', 'forward_1', 'backward_2', 'forward_2']):
            self.deform_align[module] = SecondOrderDeformableAlignment(
                2 * mid_channels, mid_channels, 3, padding=1, deform_groups=16,
                max_residue_magnitude=max_residue_magnitude)
            self.backbone[module] = ResidualBlocksWithInputConv((2 + i) * mid_channels, mid_channels, num_blocks)
        self.reconstruction = ResidualBlocksWithInputConv(5 * mid_channels, mid_channels, 5)
        self.upsample1 = PixelShufflePack(mid_channels, mid_channels, 2, upsample_kernel=3)
        self.upsample2 = PixelShufflePack(mid_channels, 64, 2, upsample_kernel=3)
        self.conv_hr = nn.Conv2d(64, 64, 3, 1, 1)
        self.conv_last = nn.Conv2d(64, 3, 3, 1, 1)
        self.img_upsample = nn.Upsample(scale_factor=4, mode='bilinear', align_corners=False)
        self.lrelu = nn.LeakyReLU(negative_slope=0.1, inplace=True)

    def compute_flow(self, lqs):
        n, t, c, h, w = lqs.size()
        a = lqs[:, :-1].reshape(-1, c, h, w)
        b = lqs[:, 1:].reshape(-1, c, h, w)
        backward = self.spynet(a, b).view(n, t - 1, 2, h, w)
        forward = self.spynet(b, a).view(n, t - 1, 2, h, w)
        return forward, backward

    def propagate(self, feats, flows, module_name):
        n, t, _, h, w = flows.size()
        frame_idx = range(0, t + 1)
        flow_idx = range(-1, t)
        mapping_idx = list(range(0, len(feats['spatial'])))
        mapping_idx += mapping_idx[::-1]
        if 'backward' in module_name:
            frame_idx = frame_idx[::-1]
            flow_idx = frame_idx
        feat_prop = flows.new_zeros(n, self.mid_channels, h, w)
        for i, idx in enumerate(frame_idx):
            feat_current = feats['spatial'][mapping_idx[idx]]
            if i > 0:
                flow_n1 = flows[:, flow_idx[i]]
                cond_n1 = flow_warp(feat_prop, flow_n1.permute(0, 2, 3, 1))
                feat_n2 = torch.zeros_like(feat_prop)
                flow_n2 = torch.zeros_like(flow_n1)
                cond_n2 = torch.zeros_like(cond_n1)
                if i > 1:
                    feat_n2 = feats[module_name][-2]
                    flow_n2 = flows[:, flow_idx[i - 1]]
                    flow_n2 = flow_n1 + flow_warp(flow_n2, flow_n1.permute(0, 2, 3, 1))
                    cond_n2 = flow_warp(feat_n2, flow_n2.permute(0, 2, 3, 1))
                cond = torch.cat([cond_n1, feat_current, cond_n2], dim=1)
                feat_prop = torch.cat([feat_prop, feat_n2], dim=1)
                feat_prop = self.deform_align[module_name](feat_prop, cond, flow_n1, flow_n2)
            feat = [feat_current] + [feats[k][idx] for k in feats if k not in ['spatial', module_name]] + [feat_prop]
            feat_prop = feat_prop + self.backbone[module_name](torch.cat(feat, dim=1))
            feats[module_name].append(feat_prop)
        if 'backward' in module_name:
            feats[module_name] = feats[module_name][::-1]
        return feats

    def upsample(self, lqs, feats):
        outputs = []
        mapping_idx = list(range(0, len(feats['spatial'])))
        mapping_idx += mapping_idx[::-1]
        for i in range(0, lqs.size(1)):
            hr = [feats[k].pop(0) for k in feats if k != 'spatial']
            hr.insert(0, feats['spatial'][mapping_idx[i]])
            hr = self.reconstruction(torch.cat(hr, dim=1))
            hr = self.lrelu(self.upsample1(hr))
            hr = self.lrelu(self.upsample2(hr))
            hr = self.lrelu(self.conv_hr(hr))
            hr = self.conv_last(hr)
            outputs.append(hr + self.img_upsample(lqs[:, i]))
        return torch.stack(outputs, dim=1)

    def forward(self, lqs):
        n, t, c, h, w = lqs.size()
        if t < 2:
            raise ValueError('BasicVSR++ needs at least two frames')
        feats = {}
        f = self.feat_extract(lqs.view(-1, c, h, w)).view(n, t, -1, h, w)
        feats['spatial'] = [f[:, i] for i in range(t)]
        flows_forward, flows_backward = self.compute_flow(lqs)
        for it in [1, 2]:
            for direction in ['backward', 'forward']:
                module = f'{direction}_{it}'
                feats[module] = []
                flows = flows_backward if direction == 'backward' else flows_forward
                feats = self.propagate(feats, flows, module)
        return self.upsample(lqs, feats)
'''
(SOURCE_DIR / 'r9_vsr_arch.py').write_text(SOURCE, encoding='utf-8')
print('Wrote', 'r9_vsr_arch.py')

## 7. Sequence engine

In [ ]:
SOURCE = r'''"""Sequence experiments."""
from dataclasses import dataclass, replace
import time
import cv2
import numpy as np
from video_core import (seed, down, base_image, detail, blocks, expand, remap, gray,
                        feature_maps, FEATURES, matlab_bicubic_down)
from r9_core import (MethodConfig, PredictableMixture, NormalMixtureCS, contrast_bounds,
    envelope_lower, certificate_lower, policy_distribution, subgaussian_proxy, exact_variance,
    variance_quadratic, best_fixed_mixture, dual_audit_ray_gaze)


@dataclass(frozen=True)
class Geometry:
    height: int = 256
    width: int = 448
    factor: int = 4
    block: int = 8
    fovea: int = 64
    frames: int = 100
    stride: int = 1
    budget: int = 1
    seed: int = 20260922
    preview_kernel: str = 'area'

    def __post_init__(self):
        for dim in [self.height, self.width]:
            assert dim % self.fovea == 0 and dim % self.factor == 0 and dim % self.block == 0
        assert self.fovea % self.factor == 0 and self.fovea % self.block == 0
        assert self.budget == 1 and self.frames >= 2
        assert self.preview_kernel in ['area', 'bicubic']


@dataclass(frozen=True)
class Method:
    name: str
    policy: str = 'joint'
    control: str = 'online'
    memory_mode: str = 'pem'
    neural: str = ''
    envelope: bool = True
    adapt_moment: bool = True
    adapt_weights: bool = True
    certificate: bool = True
    twin_audit: bool = False


def core_methods():
    """Equal-budget controlled baselines plus direct R8 and R9 ablations."""
    return [
        Method('PEM_uniform', 'uniform', 'learned', envelope=False, adapt_moment=False, adapt_weights=False, certificate=False),
        Method('PEM_greedy', 'greedy', 'learned', envelope=False, adapt_moment=False, adapt_weights=False, certificate=False),
        Method('PEM_softmax', 'softmax', 'learned', envelope=False, adapt_moment=False, adapt_weights=False, certificate=False),
        Method('PEM_risk_only', 'risk_only', 'learned', envelope=False, adapt_moment=False, adapt_weights=False, certificate=False),
        Method('PEM_audit_only', 'audit_only', 'learned', envelope=False, adapt_moment=False, adapt_weights=False, certificate=False),
        Method('PEM_R6_fixed', 'joint', 'learned', envelope=False, adapt_moment=False, adapt_weights=False, certificate=False),
        Method('PPAT2026_M1', 'ppat_2026_m1', 'learned', envelope=False, adapt_moment=False, adapt_weights=False, certificate=False),
        Method('PEM_R7_reserve', 'joint', 'online', envelope=True, certificate=False),
        # R8 predecessor.
        Method('PEM_R8_certified', 'joint', 'online', envelope=False, certificate=True, twin_audit=True),
        # Twin audit measured but not used for gaze.
        Method('PEM_R9_single_audit', 'joint', 'online', envelope=False, certificate=True, twin_audit=True),
        # Dual audit with fixed proxy weights.
        Method('PEM_R9_dual_fixed', 'dual_joint', 'learned', envelope=False, adapt_moment=False, adapt_weights=False, certificate=True, twin_audit=True),
        # Full TwinAudit method.
        Method('PEM_R9_TwinAudit', 'dual_joint', 'online', envelope=False, certificate=True, twin_audit=True),
    ]


def neural_methods(causal_names, offline_names=()):
    """Published SR baselines inside the identical one-patch sensor interface.

    _patch: current SR prediction plus the paid foveal patch (uniform gaze).
    _memory: the same plus the identical causal residual memory.
    _memory_ours: the same SR backbone controlled by the proposed dual-audit policy.
    Offline (bidirectional) video SR reads future previews; it is reported only as a
    non-causal reference and never as a deployable competitor.
    """
    out=[]
    for n in causal_names:
        out.extend([
            Method(n+'_patch','uniform','learned','sr',n,False,False,False,False,False),
            Method(n+'_memory','uniform','learned','sr_memory',n,False,False,False,False,True),
            Method(n+'_memory_R9','dual_joint','online','sr_memory',n,False,True,True,True,True),
        ])
    for n in offline_names:
        out.append(Method(n+'_patch','uniform','learned','sr',n,False,False,False,False,False))
    return out

def sensor_parameters(regime, t, total, tile=None):
    if regime == 'clean': return 0., 0., 0., 0.
    if regime == 'nominal': return .005, .01, 0., 0.
    if regime == 'noisy': return .012, .025, 0., 0.
    if regime in ['drift', 'biased_stress']:
        if t < total // 3: p, f, blur = .003, .008, 0.
        elif t < 2 * total // 3: p, f, blur = .030, .060, .8
        else: p, f, blur = .010, .020, 0.
        if tile is not None:
            f *= .5 + (tile % 5) / 4
        bias = .05 if regime == 'biased_stress' and t >= total // 3 else 0.
        return p, f, blur, bias
    raise ValueError(regime)


def preview_frame(frame, cfg, name, regime, draw, t, total):
    """The coarse noisy preview p_t. Shared by the controller and SR baselines so
    every method sees exactly the same low-resolution input."""
    ps, _, blur, _ = sensor_parameters(regime, t, total)
    optical = cv2.GaussianBlur(frame, (0, 0), blur) if blur else frame
    if cfg.preview_kernel == 'area':
        p = down(optical, cfg.factor)
    else:
        # Antialiased MATLAB-style bicubic: the native degradation of BI-trained SR checkpoints.
        p = matlab_bicubic_down(optical, cfg.factor)
    rng = np.random.default_rng(seed(cfg.seed, name, regime, draw, t, 'preview'))
    return (p + rng.normal(0, ps, p.shape).astype(np.float32)).astype(np.float32)


def previews(frames, cfg, name, regime, draw):
    return [preview_frame(f, cfg, name, regime, draw, t, len(frames)) for t, f in enumerate(frames)]


def observations(frames, cfg, name, regime, draw):
    """Only preview pixels enter optical flow and preview-derived features."""
    result, previous = [], None
    for t, frame in enumerate(frames):
        start = time.perf_counter()
        p = preview_frame(frame, cfg, name, regime, draw, t, len(frames))
        b = base_image(p, cfg.factor)
        g = np.clip(cv2.GaussianBlur(gray(b), (0, 0), 1) * 255, 0, 255).astype(np.uint8)
        if previous is None:
            flow = np.zeros((*g.shape, 2), np.float32)
            fb = np.zeros(g.shape, np.float32); innovation = fb.copy()
        else:
            flow = cv2.calcOpticalFlowFarneback(g, previous[0], None, .5, 3, 21, 3, 5, 1.2, 0)
            reverse = cv2.calcOpticalFlowFarneback(previous[0], g, None, .5, 3, 21, 3, 5, 1.2, 0)
            fb = np.sum((flow + remap(reverse, flow)) ** 2, axis=2)
            innovation = np.mean((b - remap(previous[1], flow)) ** 2, axis=2)
        pg = gray(p); rr = pg - cv2.GaussianBlur(pg, (0, 0), .7)
        noise = max(float(np.median(abs(rr - np.median(rr))) / .67448975), .0005)
        result.append(dict(p=p, b=b, flow=flow, fb=fb, innovation=innovation, noise=noise,
                           preview_ms=1000*(time.perf_counter()-start)))
        previous = (g, b)
    return result


class PatchSensor:
    def __init__(self, frames, cfg, name, regime, draw):
        self._frames = frames; self.cfg = cfg; self.name = name
        self.regime = regime; self.draw = draw; self.reads = 0
        self._seen = set()

    def acquire(self, t, action):
        c = self.cfg; s = c.fovea; nx = c.width // s; j = c.height // s * nx
        if t in self._seen or not 0 <= action < j:
            raise ValueError('One valid foveal query per frame is required')
        self._seen.add(t); self.reads += 1
        y, x = action // nx * s, action % nx * s
        sig, bias = sensor_parameters(self.regime, t, len(self._frames), action)[1::2]
        rng = np.random.default_rng(seed(c.seed, self.name, self.regime, self.draw, t, action, 'fovea'))
        clean = self._frames[t, y:y+s, x:x+s]
        # Keep the noisy observation UNCLIPPED for the cancellation identity.
        patch = (clean + bias + rng.normal(0, sig, clean.shape)).astype(np.float32)
        return patch, (y, x)


def tile_features(X, pred, base, risk, cfg):
    h, w, k, s = cfg.height, cfg.width, cfg.block, cfg.fovea
    x = X.reshape(h // s, s // k, w // s, s // k, -1)
    energy = blocks((pred - base) ** 2, s)
    risk = blocks(expand(risk, k), s)
    extra = np.stack([np.log1p(energy / 1e-6), np.log1p(np.maximum(risk, 0) / 1e-6)], axis=-1)
    return np.concatenate([x.mean((1, 3)), x.std((1, 3)), extra], axis=-1).reshape(-1, 2*len(FEATURES)+2)


class Controller:
    """Controller with a one-step counterfactual shadow memory.

    The deployed memory is always updated with the paid patch.  Separately, we
    retain a copy of the *pre-write* state and propagate it for exactly one
    frame.  At t+1 this produces a shadow prediction that answers a concrete
    counterfactual: what would the current prediction have been if the previous
    foveal write had been omitted, holding the executed gaze history and the
    current preview fixed?  The shadow never influences the deployed memory.
    """
    def __init__(self,cfg,method,memory,auditor):
        self.cfg,self.method,self.memory,self.auditor=cfg,method,memory,auditor
        self.bank=np.zeros((cfg.height,cfg.width,3),np.float32)
        self.age=np.full((cfg.height,cfg.width),24.,np.float32)
        self.hist=np.zeros_like(self.age)
        self.anchor=None
        self.shadow_source=None

    def _advance_arrays(self,bank,age,hist,anchor,o):
        bank=remap(bank,o['flow'])
        anchor=remap(anchor,o['flow'])
        age=np.minimum(remap(age,o['flow'])+1,100)
        hist=remap(hist,o['flow'])+np.maximum(o['innovation']-2*o['noise']**2,0)
        return bank,age,hist,anchor

    def _predict_arrays(self,o,bank,age,hist,anchor,sr=None):
        c=self.cfg
        X,hh,H,A,E,G,AE,V=feature_maps(bank,age,anchor,hist,o,c)
        if self.memory is None:
            gain,risk=np.exp(-A/12),G+H
        else:
            gain,risk=self.memory.predict(X,H.ravel()); gain=gain.reshape(H.shape); risk=risk.reshape(H.shape)
        base=np.clip(o['b'],0,1)
        if self.method.memory_mode=='pem':
            pred=np.clip(o['b']+expand(gain,c.block)[...,None]*hh,0,1)
        elif self.method.memory_mode=='sr':
            if sr is None: raise ValueError('Neural prediction missing')
            pred=sr
        elif self.method.memory_mode=='sr_memory':
            if sr is None: raise ValueError('Neural prediction missing')
            pred=np.clip(sr+expand(np.exp(-A/12),c.block)[...,None]*bank,0,1)
        else:
            raise ValueError(self.method.memory_mode)
        return dict(X=X,h=hh,H=H,A=A,risk_map=risk,base=base,pred=pred)

    def prepare(self,obs,sr=None):
        c=self.cfg; o=obs
        if self.anchor is None:
            self.anchor=o['b'].copy()
            shadow=None
        else:
            self.bank,self.age,self.hist,self.anchor=self._advance_arrays(self.bank,self.age,self.hist,self.anchor,o)
            shadow=None
            if self.shadow_source is not None and self.method.twin_audit:
                sb,sa,sh,san=[x.copy() for x in self.shadow_source]
                sb,sa,sh,san=self._advance_arrays(sb,sa,sh,san,o)
                shadow=self._predict_arrays(o,sb,sa,sh,san,sr)
        main=self._predict_arrays(o,self.bank,self.age,self.hist,self.anchor,sr)
        X,hh,A,risk,base,pred=main['X'],main['h'],main['A'],main['risk_map'],main['base'],main['pred']
        z=tile_features(X,pred,base,risk,c)
        lo,hi,energy=contrast_bounds(base,pred,c.fovea)
        if self.auditor is None:
            experts=np.column_stack([energy*0,energy,energy*.5])
            r=energy[:,None]-experts
            cov=np.einsum('jk,jl->jkl',r,r)+1e-10*np.eye(3)[None]
        else:
            experts,cov=self.auditor.predict(z,energy)
        benefit=blocks(expand(risk*(1-np.exp(-A/8))+1e-5*(A/24)**2,c.block),c.fovea).ravel()
        if sr is not None:
            benefit=benefit+blocks((sr-base)**2,c.fovea).ravel()
        tile_risk=blocks(expand(np.maximum(risk,0),c.block),c.fovea).ravel()
        if shadow is None or self.method.memory_mode=='sr':
            shadow_pred=pred.copy(); write_valid=False
            write_lo=np.zeros_like(energy); write_hi=np.zeros_like(energy); write_energy=np.zeros_like(energy)
        else:
            shadow_pred=shadow['pred']; write_valid=True
            write_lo,write_hi,write_energy=contrast_bounds(shadow_pred,pred,c.fovea)
        snapshot=(self.bank.copy(),self.age.copy(),self.hist.copy(),self.anchor.copy())
        return dict(base=base,pred=pred,X=X,h=hh,energy=energy,features=z,experts=experts,covariance=cov,
                    benefit=benefit,risk=tile_risk,lo=lo,hi=hi,sr=sr,shadow_pred=shadow_pred,
                    write_valid=write_valid,write_lo=write_lo,write_hi=write_hi,write_energy=write_energy,
                    _prewrite_snapshot=snapshot)

    def update(self,state,patch,location):
        c=self.cfg; y,x=location; s=c.fovea; sl=np.s_[y:y+s,x:x+s]
        # This frozen copy becomes next frame's one-step no-write counterfactual.
        self.shadow_source=tuple(a.copy() for a in state['_prewrite_snapshot']) if self.method.twin_audit else None
        if self.method.memory_mode=='sr_memory':
            self.bank[sl]=patch-state['sr'][sl]
        else:
            self.bank[sl]=detail(patch,c.factor)
        self.anchor[sl]=base_image(down(patch,c.factor),c.factor)
        self.age[sl]=0; self.hist[sl]=0
        recon=state['pred'].copy(); recon[sl]=np.clip(patch,0,1)
        return recon

def ssim_rgb(x, y):
    """Mean RGB SSIM, 11x11 Gaussian window, valid interior, dynamic range 1."""
    k1, k2 = .01**2, .03**2
    blur = lambda z: cv2.GaussianBlur(z, (11, 11), 1.5)
    a, b = blur(x), blur(y)
    vx = np.maximum(blur(x*x)-a*a, 0); vy = np.maximum(blur(y*y)-b*b, 0)
    cov = blur(x*y)-a*b
    z = ((2*a*b+k1)*(2*cov+k2))/((a*a+b*b+k1)*(vx+vy+k2))
    return float(z[5:-5, 5:-5].mean())


def run_sequence(frames, obs, cfg, acfg, name, regime, draw, method, memory=None, auditor=None,
                 sr_predictions=None, sr_ms=None, collect=False, keep=False, references=None):
    """Compare methods with common preview noise, patch noise and random uniforms.

    Evaluator references may be changed independently for boundary tests.
    No reference-derived value is returned to Controller or the online learner.
    """
    if len(frames) != len(obs): raise ValueError('Frame/preview length mismatch')
    c = cfg; n = c.height // c.fovea * (c.width // c.fovea)
    control = Controller(c, method, memory, auditor)
    effective = replace(acfg, envelope_reserve=acfg.envelope_reserve if method.envelope else 0.,
                        online_moment_mix=acfg.online_moment_mix if method.adapt_moment else 0.)
    mix = PredictableMixture(effective); cs = NormalMixtureCS(acfg.alpha, acfg.cs_rho)
    write_cs = NormalMixtureCS(acfg.alpha, acfg.cs_rho)
    sensor = PatchSensor(frames, c, name, regime, draw)
    rng = np.random.default_rng(seed(acfg.seed, name, regime, draw, 'paired-uniform'))
    records, training, memory_training, example = [], [], [], None
    true_sum = 0.; write_true_sum = 0.; write_count = 0; previous_recon = previous_ref = None
    oracle_A = np.zeros((3,3), float); oracle_b = np.zeros(3, float); oracle_c = 0.
    previous_seen = np.zeros((c.height, c.width), np.float32)
    max_sig = max(sensor_parameters(regime, t, len(frames), a)[1] for t in range(len(frames)) for a in range(n))
    if max_sig > acfg.sigma_bound + 1e-12:
        raise ValueError('Sensor standard deviation exceeds declared CS bound')
    for t, o in enumerate(obs):
        start = time.perf_counter()
        sr = sr_predictions[t] if sr_predictions is not None else None
        state = control.prepare(o, sr)
        experts = state['experts']
        if method.control == 'online':
            w = mix.weights.copy()
        elif method.control in ['learned', 'energy', 'zero']:
            w = np.eye(3)[{'zero': 0, 'energy': 1, 'learned': 2}[method.control]]
        else: raise ValueError(method.control)
        mu = experts @ w
        if method.policy == 'ppat_2026_m1':
            mu = experts[:, 2] - experts[:, 2].mean()
        moment = mix.moment(state['covariance']) if method.control == 'online' else np.einsum('k,jkl,l->j', w, state['covariance'], w)
        moment = np.maximum(moment, 1e-14)
        noise_bound = 4*acfg.sigma_bound**2*state['energy']/(3*c.fovea**2)
        # Counterfactual one-step write-value audit.  Energy is a fully
        # predictable analytic proxy; the envelope moment is calibration-free.
        write_mu = state['write_energy'].astype(float)
        write_noise_bound = 4*acfg.sigma_bound**2*state['write_energy']/(3*c.fovea**2)
        write_env = np.maximum(abs(state['write_lo']-write_mu), abs(state['write_hi']-write_mu))
        write_moment = np.maximum(write_env**2 + write_noise_bound, 1e-14) if state['write_valid'] else np.zeros(n)
        certificate_ref = np.nan; certificate_gamma = np.nan
        if method.certificate:
            lower, certificate_ref, certificate_gamma = certificate_lower(
                mu, state['lo'], state['hi'], noise_bound, effective)
        elif method.envelope:
            lower = envelope_lower(mu, state['lo'], state['hi'], noise_bound, effective)
        else:
            lower = np.full(n, effective.epsilon/n)
        dual_alpha=np.nan
        if method.policy=='dual_joint':
            q,dual_alpha,ratio,write_ratio=dual_audit_ray_gaze(
                state['benefit'],moment,write_moment,lower,effective.kappa,effective.write_kappa)
        else:
            q=policy_distribution(method.policy,state['benefit'],state['risk'],moment,experts,lower,effective,write_moment)
            qref=lower/lower.sum() if lower.sum()>=1-1e-12 else lower+(1-lower.sum())/n
            ratio=float(np.sum(moment/q)/np.sum(moment/qref))
            write_ratio=(float(np.sum(write_moment/q)/np.sum(write_moment/qref))
                         if state['write_valid'] and write_moment.sum()>0 else 0.)
        assert np.isclose(q.sum(),1) and np.min(q)>=acfg.epsilon/n-1e-12
        if method.policy in ['joint','dual_joint']: assert np.all(q>=lower-1e-12)
        if method.policy=='joint': assert ratio<=acfg.kappa+1e-7
        if method.policy=='dual_joint':
            assert ratio<=acfg.kappa+1e-7
            if state['write_valid']: assert write_ratio<=acfg.write_kappa+1e-7
        variance_proxy=subgaussian_proxy(q,mu,state['lo'],state['hi'],noise_bound)
        write_variance_proxy=(subgaussian_proxy(q,write_mu,state['write_lo'],state['write_hi'],write_noise_bound)
                              if state['write_valid'] else 0.)
        action = min(int(np.searchsorted(np.cumsum(q), rng.random(), side='right')), n-1)
        patch, loc = sensor.acquire(t, action)
        y, x = loc; s = c.fovea; sl = np.s_[y:y+s, x:x+s]
        observed = float(np.mean((patch-state['base'][sl])**2-(patch-state['pred'][sl])**2))
        ipw = observed/(n*q[action])
        estimate = float(mu.mean()+(observed-mu[action])/(n*q[action]))
        ci_lo, ci_hi, center = cs.update(estimate, variance_proxy)
        if state['write_valid']:
            observed_write=float(np.mean((patch-state['shadow_pred'][sl])**2-(patch-state['pred'][sl])**2))
            write_ipw=observed_write/(n*q[action])
            write_estimate=float(write_mu.mean()+(observed_write-write_mu[action])/(n*q[action]))
            write_ci_lo,write_ci_hi,write_center=write_cs.update(write_estimate,write_variance_proxy)
        else:
            observed_write=write_ipw=write_estimate=np.nan
            write_ci_lo=write_ci_hi=write_center=np.nan
        pp_mu = experts[:, 2] - experts[:, 2].mean()
        ppat = float((observed-pp_mu[action])/(n*q[action]))
        if collect:
            training.append((state['features'][action].copy(), observed, state['energy'][action], name))
            k = c.block
            X = state['X'].reshape(c.height//k, c.width//k, -1)[y//k:(y+s)//k, x//k:(x+s)//k].reshape(-1, len(FEATURES))
            h = state['h'][sl]; residual = patch-o['b'][sl]
            memory_training.append((X.copy(), blocks(h*residual,k).ravel(), blocks(h*h,k).ravel(), blocks(residual*residual,k).ravel()))
        recon = control.update(state, patch, loc)
        if method.control == 'online': mix.update(experts, q, action, observed, adapt_weights=method.adapt_weights)
        online_ms = 1000*(time.perf_counter()-start)
        # Everything below this point is evaluator-only.
        ref = (references if references is not None else frames)[t]
        d = blocks((ref-state['base'])**2-(ref-state['pred'])**2, s).ravel().astype(float)
        sigma = np.array([sensor_parameters(regime,t,len(frames),a)[1] for a in range(n)])
        noise_var = 4*sigma*sigma*state['energy']/(3*s*s)
        beta = np.array([sensor_parameters(regime,t,len(frames),a)[3] for a in range(n)])
        mean_observed = d + 2*beta*blocks(state['pred']-state['base'],s).ravel()
        qa, qb, qc = variance_quadratic(experts, mean_observed, q, noise_var)
        oracle_A += qa; oracle_b += qb; oracle_c += qc
        true_sum += float(d.mean()); target = true_sum/(t+1)
        if state['write_valid']:
            write_d=blocks((ref-state['shadow_pred'])**2-(ref-state['pred'])**2,s).ravel().astype(float)
            write_noise_var=4*sigma*sigma*state['write_energy']/(3*s*s)
            write_mean_observed=write_d+2*beta*blocks(state['pred']-state['shadow_pred'],s).ravel()
            write_true_sum+=float(write_d.mean()); write_count+=1
            write_target=write_true_sum/write_count
            exact_write=exact_variance(write_mean_observed,write_mu,q,write_noise_var)
            write_bias=float(write_mean_observed.mean()-write_d.mean())
            write_covered=bool(write_ci_lo<=write_target<=write_ci_hi)
        else:
            write_d=np.zeros(n); write_target=np.nan; exact_write=np.nan; write_bias=np.nan; write_covered=True
        mse = float(np.mean((recon-ref)**2)); bias = float(q@mean_observed-d.mean())
        seen = remap(previous_seen, o['flow']) > .5
        seen[sl] = False
        past_mse = float(np.mean((recon[seen]-ref[seen])**2)) if seen.any() else np.nan
        temporal = np.nan
        if previous_recon is not None:
            temporal = float(np.mean(((recon-previous_recon)-(ref-previous_ref))**2))
        row = dict(frame=t, action=action, propensity=float(q[action]), min_q=float(q.min()),
            effective_actions=float(1/(q@q)), predicted_moment_ratio=ratio,
            write_predicted_moment_ratio=float(write_ratio), dual_reference_mix=float(dual_alpha),
            certificate_reference=float(certificate_ref), certificate_gamma=float(certificate_gamma),
            certified_importance_actual=float(np.max((np.maximum(abs(state['lo']-mu), abs(state['hi']-mu)) + np.sqrt(noise_bound))/(n*q))),
            true_contrast=float(d.mean()), cumulative_true_contrast=target,
            write_valid=bool(state['write_valid']), write_true=float(write_d.mean()) if state['write_valid'] else np.nan,
            cumulative_write_true=float(write_target) if state['write_valid'] else np.nan,
            observed_write=observed_write, write_ipw=write_ipw, write_augmented=write_estimate,
            observed=observed, naive=observed, ipw=ipw, augmented=estimate, plugin=float(mu.mean()), ppat_m1=ppat,
            mse=mse, psnr=-10*np.log10(max(mse,1e-12)), ssim=ssim_rgb(recon, ref),
            preview_mse=float(np.mean((state['base']-ref)**2)),
            pre_mse=float(np.mean((state['pred']-ref)**2)), past_fovea_mse=past_mse,
            past_fovea_pixels=int(seen.sum()), temporal_residual_mse=temporal,
            exact_variance_augmented=exact_variance(mean_observed,mu,q,noise_var),
            exact_variance_ipw=exact_variance(mean_observed,np.zeros(n),q,noise_var),
            exact_variance_energy=exact_variance(mean_observed,experts[:,1],q,noise_var),
            exact_variance_learned=exact_variance(mean_observed,experts[:,2],q,noise_var),
            exact_variance_ppat=exact_variance(mean_observed,pp_mu,q,noise_var),
            exact_variance_write=exact_write, naive_conditional_bias=bias,
            augmented_conditional_bias=float(mean_observed.mean()-d.mean()), write_conditional_bias=write_bias,
            cs_low=ci_lo, cs_high=ci_hi, cs_center=center,
            write_cs_low=write_ci_lo, write_cs_high=write_ci_hi, write_cs_center=write_center,
            write_cs_covered=write_covered, write_cs_width=(max(write_ci_hi-write_ci_lo,0) if state['write_valid'] else np.nan),
            cs_covered=bool(ci_lo<=target<=ci_hi), cs_width=max(ci_hi-ci_lo,0), cs_empty=bool(ci_lo>ci_hi),
            cs_variance_proxy=variance_proxy, weight_zero=w[0], weight_energy=w[1], weight_learned=w[2],
            controller_ms=online_ms, preview_ms=o['preview_ms'], sr_ms=float(sr_ms[t]) if sr_ms is not None else 0.)
        records.append(row)
        previous_seen = remap(previous_seen, o['flow']); previous_seen[sl] = 1
        previous_recon, previous_ref = recon.copy(), ref.copy()
        if keep and t == len(frames)-1:
            example = dict(reference=ref, reconstruction=recon, preview=state['base'],
                gaze=q.reshape(c.height//s,c.width//s), weights=w.copy(), frame=t,
                audit_true=d.reshape(c.height//s,c.width//s), audit_proxy=mu.reshape(c.height//s,c.width//s),
                shadow=state['shadow_pred'], write_true=(write_d.reshape(c.height//s,c.width//s) if state['write_valid'] else None))
    assert sensor.reads == len(frames)
    import pandas as pd
    table = pd.DataFrame(records); total = len(table)
    summary = dict(frames=total, mse=float(table.mse.mean()),
        psnr_frame_mean=float(table.psnr.mean()), psnr_from_mean_mse=float(-10*np.log10(max(table.mse.mean(),1e-12))),
        ssim=float(table.ssim.mean()), preview_mse=float(table.preview_mse.mean()),
        pre_mse=float(table.pre_mse.mean()), temporal_residual_mse=float(table.temporal_residual_mse.mean()),
        past_fovea_mse=float(table.past_fovea_mse.mean()) if table.past_fovea_mse.notna().any() else None,
        acquisition_fraction=1/c.factor**2+c.fovea**2/(c.height*c.width),
        audit_true=float(table.true_contrast.mean()),
        cs_simultaneous_covered=bool(table.cs_covered.all()), cs_final_covered=bool(table.cs_covered.iloc[-1]),
        cs_final_width=float(table.cs_width.iloc[-1]), cs_valid_assumptions=regime!='biased_stress',
        cs_empty_frames=int(table.cs_empty.sum()),
        controller_ms_mean=float(table.controller_ms.mean()),
        online_total_ms_mean=float((table.controller_ms+table.preview_ms+table.sr_ms).mean()),
        online_total_ms_p95=float((table.controller_ms+table.preview_ms+table.sr_ms).quantile(.95)),
        max_predicted_moment_ratio=float(table.predicted_moment_ratio.max()),
        max_write_predicted_moment_ratio=float(table.write_predicted_moment_ratio.max()),
        dual_reference_mix_mean=float(table.dual_reference_mix.dropna().mean()) if table.dual_reference_mix.notna().any() else None,
        min_propensity=float(table.min_q.min()), effective_actions_mean=float(table.effective_actions.mean()))
    for est in ['naive','ipw','augmented','plugin','ppat_m1']:
        value = float(table[est].mean())
        summary[est+'_estimate'] = value
        summary[est+'_error'] = value-summary['audit_true']
        summary[est+'_squared_error'] = (value-summary['audit_true'])**2
    for est in ['augmented','ipw','energy','learned','ppat']:
        # Realized predictable quadratic variation / T^2. Its expectation is
        # the MSE of the martingale audit error; it is NOT the variance
        # conditional on the entire realized future trajectory.
        summary[est+'_pqv'] = float(table['exact_variance_'+est].sum()/total**2)
    wt=table[table.write_valid.astype(bool)]
    if len(wt):
        summary['write_true']=float(wt.write_true.mean())
        summary['write_estimate']=float(wt.write_augmented.mean())
        summary['write_error']=summary['write_estimate']-summary['write_true']
        summary['write_squared_error']=summary['write_error']**2
        summary['write_pqv']=float(wt.exact_variance_write.sum()/len(wt)**2)
        summary['write_cs_simultaneous_covered']=bool(wt.write_cs_covered.all())
        summary['write_cs_final_width']=float(wt.write_cs_width.iloc[-1])
        summary['write_frames']=int(len(wt))
    else:
        summary.update(write_true=0.,write_estimate=0.,write_error=0.,write_squared_error=0.,write_pqv=0.,
                       write_cs_simultaneous_covered=True,write_cs_final_width=0.,write_frames=0)
    summary['naive_mean_conditional_bias'] = float(table.naive_conditional_bias.mean())
    summary['augmented_mean_conditional_bias'] = float(table.augmented_conditional_bias.mean())
    oracle_w, oracle_total = best_fixed_mixture(oracle_A, oracle_b, oracle_c)
    oracle_pqv = oracle_total / total**2
    summary['hindsight_best_fixed_pqv'] = float(oracle_pqv)
    summary['online_mixture_regret_pqv'] = float(summary['augmented_pqv'] - oracle_pqv)
    summary['hindsight_weight_zero'] = float(oracle_w[0])
    summary['hindsight_weight_energy'] = float(oracle_w[1])
    summary['hindsight_weight_learned'] = float(oracle_w[2])
    finite_cert = table.certificate_gamma[np.isfinite(table.certificate_gamma)]
    summary['certificate_gamma_max'] = float(finite_cert.max()) if len(finite_cert) else None
    summary['certified_importance_actual_max'] = float(table.certified_importance_actual.max())
    return summary, table, training, memory_training, example
'''
(SOURCE_DIR / 'r9_engine.py').write_text(SOURCE, encoding='utf-8')
print('Wrote', 'r9_engine.py')

## 8. Statistics, tables and figures

In [ ]:
SOURCE = r'''"""Statistics and figures."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

CONTEXT = ['dataset','regime','height','width','factor','fovea','preview_kernel','study']
METRICS = ['mse','ssim','augmented_pqv','augmented_squared_error','write_pqv','write_squared_error','cs_final_width','write_cs_final_width','online_total_ms_mean']


DISPLAY = {
    'PEM_R9_TwinAudit': 'TwinAudit (ours)',
    'PEM_R9_TwinAudit_strict': 'TwinAudit-strict (κw = 1)',
    'PEM_R9_dual_fixed': 'Ours w/o online proxy mixture',
    'PEM_R9_single_audit': 'Ours w/o write-audit constraint (duplicate definition)',
    'PEM_R8_certified': 'Ours w/o write-audit constraint',
    'PEM_R7_reserve': 'Envelope-reserve joint (superseded)',
    'PEM_R6_fixed': 'Joint gaze, fixed proxy (prior version)',
    'PEM_uniform': 'Uniform gaze', 'PEM_greedy': 'Greedy gaze', 'PEM_softmax': 'Benefit-softmax gaze',
    'PEM_risk_only': 'Risk-only gaze', 'PEM_audit_only': 'Audit-only gaze',
    'PPAT2026_M1': 'PPAT (2026), M=1',
}
_SR = {'NanoSR2025': 'NanoSR (NTIRE 2025)', 'SPANF2025': 'SPAN-F (NTIRE 2025)', 'SCMSR2025': 'SCMSR (NTIRE 2025)',
       'BasicVSRpp2022_causal': 'BasicVSR++ causal (CVPR 2022)', 'BasicVSRpp2022_offline': 'BasicVSR++ offline, non-causal (CVPR 2022)'}
_SUFFIX = {'_patch': ' + patch', '_memory': ' + patch + memory', '_memory_R9': ' + TwinAudit gaze (ours)'}


def display_name(name):
    """Reader-facing method names; internal revision labels never reach figures or tables."""
    if name in DISPLAY: return DISPLAY[name]
    for suffix in ['_memory_R9', '_memory', '_patch']:
        if name.endswith(suffix) and name[:-len(suffix)] in _SR:
            return _SR[name[:-len(suffix)]] + _SUFFIX[suffix]
    for prefix, label in [('PEM_R9_writek', 'Ours, write kappa='), ('PEM_R9_cert', 'Ours, certificate kappa='),
                          ('PEM_R9_k', 'Ours, kappa=')]:
        if name.startswith(prefix): return label + name[len(prefix):]
    return name


def bootstrap_diff(a,b,rng,reps=6000):
    a,b=np.asarray(a,float),np.asarray(b,float); d=a-b
    idx=rng.integers(0,len(d),(reps,len(d)))
    return float(d.mean()),*np.quantile(d[idx].mean(1),[.025,.975])


def bootstrap_ratio(a,b,rng,reps=6000):
    a,b=np.asarray(a,float),np.asarray(b,float)
    if len(a)<2 or not np.isfinite(a).all() or not np.isfinite(b).all():
        raise ValueError('Paired inference needs finite data from at least two source clusters')
    if b.mean()<=1e-25:
        return None
    values=[]
    for start in range(0,reps,256):
        idx=rng.integers(0,len(a),(min(256,reps-start),len(a)))
        den=b[idx].mean(1)
        if np.any(den<=1e-25):
            return None
        values.extend(100*(1-a[idx].mean(1)/den))
    return 100*(1-a.mean()/b.mean()),*np.quantile(values,[.025,.975])


def paired_signflip(a,b,rng,reps=20000):
    """Requires exchangeability/symmetry of paired source effects under H0."""
    delta=np.asarray(b)-np.asarray(a);obs=abs(delta.mean());n=len(delta)
    if n<=14:
        ids=np.arange(2**n,dtype=np.uint32)
        signs=2*((ids[:,None]>>np.arange(n))&1).astype(float)-1
        return float(np.mean(abs(signs@delta/n)>=obs-1e-15))
    count=0
    for start in range(0,reps,256):
        signs=rng.choice([-1,1],(min(256,reps-start),n))
        count+=int(np.sum(abs(signs@delta/n)>=obs-1e-15))
    return (count+1)/(reps+1)


def holm(p):
    p=np.asarray(p,float);order=np.argsort(p)
    adjusted=np.minimum(1,np.maximum.accumulate(p[order]*(len(p)-np.arange(len(p)))))
    out=np.empty(len(p));out[order]=adjusted;return out


def paired_statistics(df,reps=6000,target='PEM_R9_TwinAudit'):
    rng=np.random.default_rng(27182);rows=[]
    match=['clip_id','source_group','draw']
    for context,g in df.groupby(CONTEXT,dropna=False,sort=True):
        if target not in set(g.method):continue
        for baseline in sorted(set(g.method)-{target}):
            aa=g[g.method==target].set_index(match)
            bb=g[g.method==baseline].set_index(match)
            if set(aa.index)!=set(bb.index):
                raise AssertionError(f'Incomplete paired comparisons: {context}/{baseline}')
            for metric in METRICS:
                a=aa[metric].groupby('source_group').mean().sort_index()
                b=bb[metric].groupby('source_group').mean().sort_index()
                if len(a)<2:continue
                stat=bootstrap_ratio(a.values,b.values,rng,reps)
                if stat is None:continue
                row=dict(zip(CONTEXT,context));row.update(target=target,baseline=baseline,metric=metric,
                    source_clusters=len(a),clips=g.clip_id.nunique(),
                    reduction_pct=stat[0],ci_low=stat[1],ci_high=stat[2],
                    target_mean=float(a.mean()),baseline_mean=float(b.mean()),
                    paired_p=paired_signflip(a.values,b.values,rng),
                    positive_reduction_favors_target=metric!='ssim',effect_type='percent_reduction')
                rows.append(row)
            # PSNR (dB) from source-mean MSE: standard reporting unit in SR literature.
            a=-10*np.log10(np.maximum(aa['mse'].groupby('source_group').mean().sort_index(),1e-12))
            b=-10*np.log10(np.maximum(bb['mse'].groupby('source_group').mean().sort_index(),1e-12))
            if len(a)>=2:
                point,lo,hi=bootstrap_diff(a.values,b.values,rng,reps)
                row=dict(zip(CONTEXT,context));row.update(target=target,baseline=baseline,metric='psnr_db',
                    source_clusters=len(a),clips=g.clip_id.nunique(),reduction_pct=point,ci_low=lo,ci_high=hi,
                    target_mean=float(a.mean()),baseline_mean=float(b.mean()),
                    paired_p=paired_signflip(a.values,b.values,rng),
                    positive_reduction_favors_target=True,effect_type='difference_dB')
                rows.append(row)
    out=pd.DataFrame(rows)
    if len(out):
        out['holm_p']=np.nan
        for _,idx in out.groupby(['study','metric']).groups.items():
            out.loc[idx,'holm_p']=holm(out.loc[idx,'paired_p'].values)
    return out


BACKBONE_METRICS = ['mse','augmented_pqv','write_pqv']


def backbone_statistics(df,reps=6000):
    """Same-backbone gaze comparisons in the neural studies.

    Added after the laptop pilot and before the paper run. For each published
    backbone B: B_memory_R9 (TwinAudit gaze) vs B_memory (uniform gaze, identical
    memory) and vs B_patch (uniform gaze, no memory). Same paired source-cluster
    bootstrap and sign-flip test as paired_statistics; Holm within study and metric.
    Ratios with a zero baseline (write PQV of _patch, which has no write audit)
    are undefined and omitted, as in paired_statistics.
    """
    rng=np.random.default_rng(31415);rows=[]
    match=['clip_id','source_group','draw']
    neural=df[df.study.astype(str).str.startswith('neural')]
    for context,g in neural.groupby(CONTEXT,dropna=False,sort=True):
        methods=set(g.method)
        for target in sorted(m for m in methods if m.endswith('_memory_R9')):
            backbone=target[:-len('_memory_R9')]
            for baseline in [backbone+'_memory',backbone+'_patch']:
                if baseline not in methods:continue
                aa=g[g.method==target].set_index(match)
                bb=g[g.method==baseline].set_index(match)
                if set(aa.index)!=set(bb.index):
                    raise AssertionError(f'Incomplete paired comparisons: {context}/{target}/{baseline}')
                common=dict(zip(CONTEXT,context));common.update(backbone=backbone,target=target,baseline=baseline)
                for metric in BACKBONE_METRICS:
                    a=aa[metric].groupby('source_group').mean().sort_index()
                    b=bb[metric].groupby('source_group').mean().sort_index()
                    if len(a)<2:continue
                    stat=bootstrap_ratio(a.values,b.values,rng,reps)
                    if stat is None:continue
                    rows.append(dict(common,metric=metric,source_clusters=len(a),clips=g.clip_id.nunique(),
                        reduction_pct=stat[0],ci_low=stat[1],ci_high=stat[2],
                        target_mean=float(a.mean()),baseline_mean=float(b.mean()),
                        paired_p=paired_signflip(a.values,b.values,rng),
                        positive_reduction_favors_target=True,effect_type='percent_reduction'))
                a=-10*np.log10(np.maximum(aa['mse'].groupby('source_group').mean().sort_index(),1e-12))
                b=-10*np.log10(np.maximum(bb['mse'].groupby('source_group').mean().sort_index(),1e-12))
                if len(a)>=2:
                    point,lo,hi=bootstrap_diff(a.values,b.values,rng,reps)
                    rows.append(dict(common,metric='psnr_db',source_clusters=len(a),clips=g.clip_id.nunique(),
                        reduction_pct=point,ci_low=lo,ci_high=hi,
                        target_mean=float(a.mean()),baseline_mean=float(b.mean()),
                        paired_p=paired_signflip(a.values,b.values,rng),
                        positive_reduction_favors_target=True,effect_type='difference_dB'))
    out=pd.DataFrame(rows)
    if len(out):
        out['holm_p']=np.nan
        for _,idx in out.groupby(['study','metric']).groups.items():
            out.loc[idx,'holm_p']=holm(out.loc[idx,'paired_p'].values)
        out['target_display']=out.target.map(display_name);out['baseline_display']=out.baseline.map(display_name)
    return out


def export_backbone_comparisons(df,out,synthetic=False,reps=6000):
    """CSV + booktabs LaTeX for backbone_statistics (post-pilot amendment)."""
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    table=backbone_statistics(df,reps)
    t=table.copy();t['evidence_scope']='SYNTHETIC SOFTWARE CHECK ONLY' if synthetic else 'NATURAL VIDEO WITH SIMULATED SENSING'
    t['amendment']='added after the laptop pilot and before the paper run'
    t.to_csv(out/'backbone_gaze_comparisons.csv',index=False)
    if len(table):
        tab=out/'tables';tab.mkdir(exist_ok=True)
        keep=table.copy()
        keep['Effect']=np.where(keep.effect_type=='difference_dB','dB gain','% reduction')
        keep['metric']=keep.metric.map({'psnr_db':'PSNR','mse':'MSE','augmented_pqv':'Audit PQV','write_pqv':'Write PQV'})
        cols={'dataset':'Dataset','regime':'Regime','target_display':'Ours','baseline_display':'Baseline','metric':'Metric',
              'Effect':'Effect','reduction_pct':'Estimate','ci_low':'95% CI low','ci_high':'95% CI high','holm_p':'Holm p'}
        for study,h in keep.groupby('study'):
            write_latex(h[list(cols)].rename(columns=cols),tab/f'table_backbone_gaze_{study}.tex',
                f'Same-backbone gaze comparisons ({study}): each published backbone with TwinAudit gaze vs the same backbone with uniform gaze (+ memory, + patch). Positive values favour TwinAudit gaze; paired source-cluster 95% percentile bootstrap intervals; Holm-adjusted sign-flip p values. Comparison added after the laptop pilot, before the paper run.',
                f'tab:backbone_gaze_{study}')
    return table


def control_statistics(df,reps=6000):
    """Matched estimator ablations on the very same executed R9 trajectories."""
    rng=np.random.default_rng(9821);rows=[]
    for context,g in df[df.method=='PEM_R9_TwinAudit'].groupby(CONTEXT,sort=True):
        p=g.groupby('source_group').mean(numeric_only=True)
        if len(p)<2:continue
        for est in ['ipw','energy','learned','ppat']:
            stat=bootstrap_ratio(p.augmented_pqv,p[est+'_pqv'],rng,reps)
            if stat is None:continue
            row=dict(zip(CONTEXT,context));row.update(comparator=est,source_clusters=len(p),
                reduction_pct=stat[0],ci_low=stat[1],ci_high=stat[2],
                paired_p=paired_signflip(p.augmented_pqv.values,p[est+'_pqv'].values,rng))
            rows.append(row)
    out=pd.DataFrame(rows)
    if len(out):out['holm_p']=holm(out.paired_p.values)
    return out


def source_summary(df):
    keys=CONTEXT+['method']
    numeric=df.select_dtypes(include=['number','bool']).columns.difference(CONTEXT+['draw'])
    source=df.groupby(keys+['source_group'])[numeric].mean().reset_index()
    group=source.groupby(keys)
    result=group[list(numeric)].mean().reset_index()
    result=result.merge(group.size().rename('source_clusters').reset_index(),on=keys)
    result=result.merge(df.groupby(keys).clip_id.nunique().rename('clips').reset_index(),on=keys)
    return result


def export_analysis(df,out,synthetic=False,reps=6000):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    summary=source_summary(df);paired=paired_statistics(df,reps);controls=control_statistics(df,reps)
    scope='SYNTHETIC SOFTWARE CHECK ONLY' if synthetic else 'NATURAL VIDEO WITH SIMULATED SENSING'
    for table in [summary]:
        if 'method' in table: table['display_name']=table.method.map(display_name)
    if len(paired):
        paired['target_display']=paired.target.map(display_name); paired['baseline_display']=paired.baseline.map(display_name)
    for name,table in [('sequence_results',df),('source_summary',summary),('paired_comparisons',paired),('same_gaze_ablation',controls)]:
        table=table.copy();table['evidence_scope']=scope;table.to_csv(out/(name+'.csv'),index=False)
    return summary,paired,controls


def publication_figures(df,summary,paired,controls,out,synthetic=False,line_dpi=1000):
    out=Path(out);out.mkdir(parents=True,exist_ok=True);paths=[];captions=[]
    plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,
                         'pdf.fonttype':42,'ps.fonttype':42,'svg.fonttype':'none'})
    def save(fig,name,caption,dpi=line_dpi):
        if synthetic:
            fig.suptitle('Synthetic software check - not benchmark evidence',fontsize=11,y=1.015)
        fig.tight_layout()
        for ext in ['pdf','svg','png']:
            fig.savefig(out/(name+'.'+ext),dpi=dpi,bbox_inches='tight',facecolor='white')
        paths.append(str(out/(name+'.png')));plt.close(fig)
        captions.append(name+'. '+caption)
    # A result-free graphical abstract: exact computation and feedback paths.
    from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
    fig,ax=plt.subplots(figsize=(11,4.5));ax.set_xlim(0,11);ax.set_ylim(0,4.5);ax.axis('off')
    nodes=[(.25,2.65,'Observe preview','Warp predictive memory'),
           (4.,2.65,'Plan gaze','Benefit + audit budget'),
           (7.75,2.65,'Acquire one patch','Log its probability'),
           (7.75,.6,'Audit memory','Correct selective sampling'),
           (4.,.6,'Update mixture','Learn from paid observation'),
           (.25,.6,'Carry memory forward','Next frame only')]
    for x,y,title,sub in nodes:
        ax.add_patch(FancyBboxPatch((x,y),3,.95,boxstyle='round,pad=.08',facecolor='#EDF3F8',edgecolor='#365F82',linewidth=1.2))
        ax.text(x+1.5,y+.61,title,ha='center',va='center',fontsize=11,fontweight='bold')
        ax.text(x+1.5,y+.26,sub,ha='center',va='center',fontsize=8.5)
    for start,end in [((3.35,3.1),(3.85,3.1)),((7.1,3.1),(7.6,3.1)),
                      ((9.25,2.5),(9.25,1.7)),((7.6,1.1),(7.1,1.1)),
                      ((3.85,1.1),(3.35,1.1)),((1.75,1.7),(1.75,2.5))]:
        ax.add_patch(FancyArrowPatch(start,end,arrowstyle='-|>',mutation_scale=14,color='#365F82',linewidth=1.3))
    ax.text(5.5,4.1,'BioSaccade-PEM TwinAudit',ha='center',fontsize=13,fontweight='bold')
    ax.text(5.5,.15,'Current audit uses pre-query weights. Dense reference images enter evaluation only.',ha='center',fontsize=9)
    save(fig,'00_graphical_abstract','Computation order and causal feedback. This schematic contains no empirical performance claim.')
    main=summary[summary.study.isin(['main','reds_train_full','reds_train_detail'])]
    if len(main):
        for (st,ds,reg),g in main.groupby(['study','dataset','regime']):
            fig,axs=plt.subplots(1,2,figsize=(10,4.4))
            for ax,metric,label in [(axs[0],'mse','RGB reconstruction MSE'),(axs[1],'augmented_pqv','Audit quadratic variation / T²')]:
                gg=g.sort_values(metric);y=np.arange(len(gg))
                colors=['#C44E52' if m=='PEM_R9_TwinAudit' else '#4C72B0' for m in gg.method]
                ax.barh(y,gg[metric],color=colors)
                ax.set_yticks(y,[display_name(m) for m in gg.method],fontsize=8)
                ax.set_xlabel(label);ax.set_title(f'{ds} ({st}) - {reg}')
                if metric=='augmented_pqv' and (gg[metric]>0).all():ax.set_xscale('log')
            save(fig,f'01_controlled_policy_{st}_{ds}_{reg}',
                 'Equal sensing budgets and common PEM backbone. Sources are averaged before aggregation; lower is better in both panels. The right panel is evaluator-only predictable quadratic variation, not an online variance certificate.')
    neural=summary[summary.study.astype(str).str.startswith('neural')]
    if len(neural):
        for (st,ds,reg),g in neural.groupby(['study','dataset','regime']):
            fig,ax=plt.subplots(figsize=(9,4.5))
            g=g.sort_values('mse');y=np.arange(len(g))
            colors=['#C44E52' if ('_R9' in m or m=='PEM_R9_TwinAudit') else ('#999999' if 'offline' in m else '#4C72B0') for m in g.method]
            ax.barh(y,g.mse,color=colors)
            ax.set_yticks(y,[display_name(m) for m in g.method],fontsize=8)
            ax.set_title(ds+' - '+('native clean bicubic input' if st=='neural_native' else 'sensor-noise input'));ax.set_xlabel('RGB reconstruction MSE (lower is better)')
            save(fig,f'02_published_backbones_{st}_{ds}_{reg}',
                 'Frozen published checkpoints (NTIRE 2025 ESR; BasicVSR++ CVPR 2022, Vimeo-90K BI) in the common one-patch sensor interface. Patch: SR output plus the paid patch; memory: identical causal residual wrapper; ours: the same backbone with the proposed gaze. Grey: offline BasicVSR++ reads future frames and is a non-causal reference only.')
    if len(controls):
        p=controls[controls.study=='main'].reset_index(drop=True)
        if len(p):
            fig,ax=plt.subplots(figsize=(9,max(3.2,.32*len(p))))
            y=np.arange(len(p));ax.hlines(y,p.ci_low,p.ci_high,color='#4C72B0');ax.scatter(p.reduction_pct,y,color='#4C72B0')
            ax.set_yticks(y,[f'{r.dataset} / {r.regime} / {r.comparator}' for r in p.itertuples()],fontsize=8)
            ax.axvline(0,color='black',linestyle='--',linewidth=1);ax.set_xlabel('R9 primary-audit variance reduction on identical trajectories (%)')
            save(fig,'03_same_gaze_proxy_ablation',
                 'Paired source-cluster 95% percentile bootstrap intervals for ratios of means. Negative values are retained. Holm adjustment applies to p values in the separate table, not these intervals.')
    sweep=summary[summary.study=='kappa']
    if len(sweep):
        for (ds,reg),g in sweep.groupby(['dataset','regime']):
            fig,ax=plt.subplots(figsize=(6,4))
            g=g.sort_values('kappa');ax.plot(g.mse,g.augmented_pqv,'o-',color='#C44E52')
            for row in g.itertuples():ax.annotate(f'κ={row.kappa:g}',(row.mse,row.augmented_pqv),xytext=(4,5),textcoords='offset points',fontsize=8)
            ax.set_xlabel('RGB reconstruction MSE');ax.set_ylabel('Audit quadratic variation / T²');ax.set_yscale('log');ax.set_title(ds+' - '+reg)
            save(fig,f'04_tradeoff_{ds}_{reg}','Prespecified budget sweep. The curve is an empirical trade-off and is not asserted to be a Pareto frontier or monotone in realized reconstruction.')
    main_r9=summary[(summary.study=='main') & (summary.method=='PEM_R9_TwinAudit')]
    if len(main_r9):
        fig,ax=plt.subplots(figsize=(7,4.5))
        labels=[f'{r.dataset} / {r.regime}' for r in main_r9.itertuples()]
        x=np.arange(len(main_r9)); width=.36
        ax.bar(x-width/2,main_r9.augmented_pqv,width,label='memory-vs-preview audit')
        ax.bar(x+width/2,main_r9.write_pqv,width,label='one-step write-value audit')
        ax.set_yscale('log');ax.set_ylabel('Predictable quadratic variation / T²')
        ax.set_xticks(x,labels,rotation=25,ha='right');ax.legend(fontsize=8)
        save(fig,'07_twin_audit_reliability','Full R9 reports two distinct design-based audits from the same paid observations: overall memory benefit and the one-step counterfactual value of the previous foveal write.')
    wk=summary[summary.study=='write_kappa']
    if len(wk):
        for (ds,reg),g in wk.groupby(['dataset','regime']):
            fig,ax=plt.subplots(figsize=(6.2,4.2));g=g.sort_values('write_kappa')
            ax.plot(g.mse,g.write_pqv,'o-')
            for row in g.itertuples():ax.annotate(f'κw={row.write_kappa:g}',(row.mse,row.write_pqv),xytext=(4,5),textcoords='offset points',fontsize=8)
            ax.set_xlabel('RGB reconstruction MSE');ax.set_ylabel('Twin-write audit PQV / T²');ax.set_yscale('log')
            ax.set_title(ds+' - '+reg)
            save(fig,f'08_write_tradeoff_{ds}_{reg}','Prespecified twin-write budget sweep; lower κw enforces stronger auditability of the one-step memory-write effect.')
    (out/'captions.txt').write_text('\n\n'.join(captions),encoding='utf-8')
    return paths


def trace_figures(trace,example,out,synthetic=False):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    fig,axs=plt.subplots(3,1,figsize=(8,8),sharex=True)
    t=trace.frame.to_numpy()+1
    axs[0].fill_between(t,trace.cs_low.to_numpy(),trace.cs_high.to_numpy(),alpha=.2,color='#4C72B0',label='95% anytime confidence sequence')
    axs[0].plot(t,trace.cs_center,label='Audit estimate',color='#4C72B0')
    axs[0].plot(t,trace.cumulative_true_contrast,label='Evaluator reference',color='black',linestyle='--')
    axs[0].set_ylabel('Cumulative average contrast');axs[0].legend(fontsize=8)
    valid=trace.write_valid.astype(bool).to_numpy()
    if valid.any():
        axs[1].plot(t[valid],trace.loc[valid,'cumulative_write_true'],label='Evaluator write value',linestyle='--')
        axs[1].plot(t[valid],trace.loc[valid,'write_cs_center'],label='Twin-write estimate')
        axs[1].fill_between(t[valid],trace.loc[valid,'write_cs_low'],trace.loc[valid,'write_cs_high'],alpha=.2)
        axs[1].legend(fontsize=8)
    axs[1].set_ylabel('One-step write value')
    for key in ['zero','energy','learned']:axs[2].plot(t,trace['weight_'+key],label=key)
    axs[2].set_xlabel('Frame');axs[2].set_ylabel('Predictable mixture weight');axs[2].legend()
    if synthetic:fig.suptitle('Synthetic software check - not benchmark evidence')
    fig.tight_layout()
    for ext in ['pdf','svg','png']:fig.savefig(out/('05_online_audit.'+ext),dpi=1000,bbox_inches='tight')
    plt.close(fig)
    if example is not None:
        fig,axs=plt.subplots(1,4,figsize=(12,3.2))
        for ax,k,title in zip(axs,['reference','preview','reconstruction','gaze'],['Reference','Coarse preview','Reconstruction','Gaze probabilities']):
            ax.imshow(example[k],cmap='viridis' if k=='gaze' else None);ax.set_title(title);ax.axis('off')
        if synthetic:fig.suptitle('Synthetic software check - not benchmark evidence')
        fig.tight_layout()
        for ext in ['pdf','png']:fig.savefig(out/('06_example.'+ext),dpi=600,bbox_inches='tight')
        plt.close(fig)
    with (out/'captions.txt').open('a',encoding='utf-8') as f:
        f.write('\n\n05_online_audit. Preselected example; conservative range-based confidence sequence for the running on-trajectory mean. Weights used at frame t are fixed before its foveal query.\n\n06_example. Preselected final frame with identical display scaling; the reference is evaluator-only. A high-DPI export does not add detail to the source image.\n')


def _latex_escape(x):
    x=str(x)
    for a,b in [('\\','\\textbackslash{}'),('&','\\&'),('%','\\%'),('_','\\_'),('#','\\#')]:
        x=x.replace(a,b)
    return x


def write_latex(table,path,caption,label):
    cols=list(table.columns)
    lines=['\\begin{table}[t]','\\centering','\\caption{'+_latex_escape(caption)+'}','\\label{'+label+'}',
           '\\small','\\begin{tabular}{'+'l'*2+'r'*(len(cols)-2)+'}','\\toprule',
           ' & '.join(_latex_escape(c) for c in cols)+' \\\\','\\midrule']
    for row in table.itertuples(index=False):
        cells=[]
        for v in row:
            if isinstance(v,(float,np.floating)):
                cells.append('--' if not np.isfinite(v) else (f'{v:.3g}' if abs(v)<1e-2 and v!=0 else f'{v:.4f}' if abs(v)<10 else f'{v:.2f}'))
            else: cells.append(_latex_escape(v))
        lines.append(' & '.join(cells)+' \\\\')
    lines+=['\\bottomrule','\\end{tabular}','\\end{table}']
    Path(path).write_text('\n'.join(lines)+'\n',encoding='utf-8')


def paper_tables(summary,paired,out,neural_provenance=None):
    """Journal-ready result tables (CSV + booktabs LaTeX). Values are means over
    source clusters; PSNR is the source-averaged per-frame PSNR."""
    out=Path(out);out.mkdir(parents=True,exist_ok=True);written={}
    cols={'display_name':'Method','regime':'Regime','source_clusters':'Sources','clips':'Clips',
          'psnr_frame_mean':'PSNR (dB)','ssim':'SSIM','augmented_pqv':'Audit PQV','write_pqv':'Write PQV'}
    summary=summary.copy()
    if 'write_frames' in summary:
        # Methods without a twin audit have no write-value estimate: show '--', not 0.
        summary.loc[summary.write_frames<=0,'write_pqv']=np.nan
    for study,g in summary.groupby('study'):
        for ds,h in g.groupby('dataset'):
            t=h.sort_values(['regime','psnr_frame_mean'],ascending=[True,False])[list(cols)].rename(columns=cols)
            name=f'table_{study}_{ds}'
            t.to_csv(out/(name+'.csv'),index=False)
            write_latex(t,out/(name+'.tex'),f'{ds}: {study} study. Means over source clusters. Audit PQV is the evaluator-side predictable quadratic variation of the memory-benefit audit divided by T^2 (lower is more reliable).',f'tab:{study}_{ds}')
            written[name]=t
    tim=summary[summary.study=='timing']
    if len(tim):
        t=tim[['display_name','online_total_ms_mean','online_total_ms_p95','controller_ms_mean']].rename(columns={
            'display_name':'Method','online_total_ms_mean':'Mean ms/frame','online_total_ms_p95':'p95 ms/frame','controller_ms_mean':'Controller ms/frame'})
        if neural_provenance:
            params={}
            for n,p in neural_provenance.items(): params[n]=p.get('parameter_count')
            t['SR parameters']=[next((params[k] for k in params if m.startswith(_SR.get(k,'\0'))),np.nan) for m in t.Method]
        t.to_csv(out/'table_latency.csv',index=False)
        write_latex(t,out/'table_latency.tex','Serial per-frame latency (no competing processes): preview/flow, controller and audit, and SR including device transfers.','tab:latency')
        written['table_latency']=t
    if len(paired):
        keep=paired[paired.metric.isin(['psnr_db','mse','augmented_pqv','write_pqv'])].copy()
        keep['Effect']=np.where(keep.effect_type=='difference_dB','dB gain','% reduction')
        keep['metric']=keep.metric.map({'psnr_db':'PSNR','mse':'MSE','augmented_pqv':'Audit PQV','write_pqv':'Write PQV'})
        t=keep[['study','dataset','regime','baseline_display','metric','Effect','reduction_pct','ci_low','ci_high','holm_p']].rename(columns={
            'baseline_display':'Baseline','reduction_pct':'Estimate','ci_low':'95% CI low','ci_high':'95% CI high','holm_p':'Holm p'})
        t.to_csv(out/'table_paired_vs_ours.csv',index=False)
        for study,h in t.groupby('study'):
            write_latex(h.drop(columns=['study']),out/f'table_paired_{study}.tex',
                        f'Paired source-cluster comparisons of TwinAudit (ours) against each baseline ({study}). Positive values favour ours; 95% percentile bootstrap intervals; Holm-adjusted sign-flip p values.',f'tab:paired_{study}')
        written['table_paired_vs_ours']=t
    return written
'''
(SOURCE_DIR / 'r9_stats.py').write_text(SOURCE, encoding='utf-8')
print('Wrote', 'r9_stats.py')

## 9. Implementation checks

In [ ]:
SOURCE = r'''"""Implementation checks."""
from dataclasses import replace
import numpy as np
from scipy.optimize import minimize
from r9_core import (MethodConfig, constrained_gaze, lower_normalize, audit_gradient,
                     exact_variance, NormalMixtureCS, fixed_pool_ppat, envelope_lower,
                     certificate_lower, policy_distribution, variance_quadratic, best_fixed_mixture,
                     subgaussian_proxy, dual_audit_ray_gaze)
from r9_engine import Geometry, Method, core_methods, observations, run_sequence
from r9_data import synthetic_frames


def dataset_reader_checks():
    """Exercise the DAVIS and REDS (val and train) directory readers with labeled image fixtures."""
    from pathlib import Path
    from tempfile import TemporaryDirectory
    import cv2
    from r9_data import discover_davis, discover_reds, validate_manifest, load_clip
    cfg=Geometry(height=64,width=96,fovea=16,frames=7)
    with TemporaryDirectory() as directory:
        root=Path(directory)
        for folder,kind,value in [(root/'DAVIS/JPEGImages/480p/a','davis',50),
                                  (root/'REDS/val_sharp/000','reds',90),
                                  (root/'REDS/val_sharp/001','reds',130),
                                  (root/'REDS/train_sharp/000','reds',180),
                                  (root/'REDS/train_sharp/001','reds',220)]:
            folder.mkdir(parents=True)
            for t in range(7):
                name=f'{t:08d}.'+('jpg' if kind=='davis' else 'png')
                assert cv2.imwrite(str(folder/name),np.full((64,96,3),value+t,np.uint8))
        ds=discover_davis(root/'DAVIS',{'fit':['a']},7)
        rs=discover_reds(root/'REDS/val_sharp',frames=7,dataset='REDS',expected=2)
        ts=discover_reds(root/'REDS/train_sharp',frames=7,dataset='REDS_train',expected=2)
        assert [c.clip_id for c in rs]==[c.clip_id for c in ts]==['000','001']
        allclips,report=validate_manifest(ds+rs+ts)
        assert len({c.source_group for c in allclips})==5, 'REDS train and val identities collided'
        for c in allclips:
            x=load_clip(c,cfg);assert x.shape==(7,64,96,3) and np.isfinite(x).all()
        try: discover_reds(root/'REDS/train_sharp',frames=7,dataset='REDS_train',expected=3)
        except ValueError: pass
        else: raise AssertionError('An incomplete external split was accepted')
        try: discover_reds(root/'REDS',frames=7,dataset='REDS')
        except ValueError: pass
        else: raise AssertionError('Mixed REDS train/val folders were accepted as one split')
        try: validate_manifest(allclips+[replace(ds[0],clip_id='alias',role='external_test')])
        except AssertionError: pass
        else: raise AssertionError('Source leakage was accepted')
    return dict(davis_reader=True,reds_val_reader=True,reds_train_reader=True,
                reds_train_val_ids_distinct=True,incomplete_split_rejected=True,
                mixed_reds_splits_rejected=True,source_leakage_rejected=True,
                scope='directory/image software fixtures, not downloaded natural benchmarks')


def mathematical_checks():
    rng = np.random.default_rng(451)
    unbiased=[]; gradient=[]; solver=[]; psd=[]; envelopes=[]; certificates=[]; quadratics=[]; dual_ratios=[]
    for _ in range(20):
        n = 12; r = rng.lognormal(size=n); v = rng.lognormal(size=n)
        lower = .05/n + .15*rng.dirichlet(np.ones(n))
        ref = lower+(1-lower.sum())/n; cap = 2*float(np.sum(v/ref))
        q = constrained_gaze(r, v, lower, 2.)
        opt = minimize(lambda p: -r@p, ref, method='SLSQP', bounds=list(zip(lower,np.ones(n))),
            constraints=[{'type':'eq','fun':lambda p:p.sum()-1},
                         {'type':'ineq','fun':lambda p:1-np.sum(v/p)/cap}],
            options={'ftol':1e-10,'maxiter':500})
        if not opt.success:
            raise AssertionError('Independent solver check failed: '+opt.message)
        solver.append(abs(float(r@q-r@opt.x)))
        assert np.all(q>=lower-1e-12) and np.sum(v/q)<=cap*(1+1e-8)
        d = rng.uniform(-.2,.2,n); M = rng.uniform(-.2,.2,(n,3)); w = np.array([.2,.3,.5]); mu=M@w
        est = mu.mean()+(d-mu)/(n*q)
        unbiased.append(abs(float(q@est-d.mean())))
        g = sum(q[a]*audit_gradient(M,w,q,a,d[a]) for a in range(n))
        numerical = []
        for k in range(3):
            e = np.eye(3)[k]*1e-5
            numerical.append((exact_variance(d,M@(w+e),q,np.zeros(n))-exact_variance(d,M@(w-e),q,np.zeros(n)))/2e-5)
        gradient.append(float(np.max(abs(g-numerical))))
        hessian = 2*(M.T@(M/q[:,None])/n**2-np.outer(M.mean(0),M.mean(0)))
        psd.append(float(np.linalg.eigvalsh(hessian).min()))
        # Analytic envelope cap remains valid even with inaccurate learned moments.
        acfg=MethodConfig(); low=d-rng.uniform(0,.3,n); high=d+rng.uniform(0,.3,n)
        tau=rng.uniform(0,.001,n); l=envelope_lower(mu,low,high,tau,acfg)
        qq=constrained_gaze(r,v,l,acfg.kappa)
        c=np.maximum(abs(low-mu),abs(high-mu))+np.sqrt(tau)
        B=c.sum()/(n*(1-acfg.epsilon)*acfg.envelope_reserve)
        envelopes.append(subgaussian_proxy(qq,mu,low,high,tau)/(2*B*B))
        assert np.max(c/(n*qq))<=B+1e-12 and envelopes[-1]<=1+1e-12
        # R9 hard certificate: independent of learned second-moment calibration.
        cl, bref, gamma = certificate_lower(mu, low, high, tau, acfg)
        cq = constrained_gaze(r, v, cl, acfg.kappa)
        certificates.append(float(np.max(c/(n*cq))/gamma))
        assert certificates[-1] <= 1 + 1e-10
        # R9 simultaneous two-audit ray allocation.  The reference is feasible
        # by construction, and the returned point must satisfy both caps.
        v2=rng.lognormal(size=n)
        q2,alpha,rr1,rr2=dual_audit_ray_gaze(r,v,v2,cl,1.7,2.3)
        assert np.isclose(q2.sum(),1) and np.all(q2>=cl-1e-12)
        assert rr1<=1.7+1e-8 and rr2<=2.3+1e-8 and 0<=alpha<=1
        dual_ratios.append((rr1,rr2,alpha))
        # Quadratic hindsight objective exactly reproduces exact_variance.
        qa,qb,qc = variance_quadratic(M,d,q,np.zeros(n))
        quadratics.append(abs(float(w@qa@w+2*qb@w+qc)-exact_variance(d,M@w,q,np.zeros(n))))
    assert max(solver)<2e-6 and max(unbiased)<1e-12 and max(gradient)<1e-8 and min(psd)>-1e-10
    n = 14; d=rng.uniform(-1,1,n); mu=rng.uniform(-1,1,n)
    result = [fixed_pool_ppat(d,mu,np.ones(n),5,rng) for _ in range(12000)]
    result = np.asarray(result); mean_error=result.mean(0)-d.mean()
    assert np.all(abs(mean_error)<5*result.std(0)/np.sqrt(len(result)))
    census=fixed_pool_ppat(d,mu,np.ones(n),n,rng)
    assert np.allclose(census,d.mean())
    # Regression test for the supplied R7 bug: risk-only must use risk, not benefit.
    cfg=MethodConfig(); benefit=np.linspace(1,10,n); risk=np.linspace(10,1,n)
    dummy=np.ones(n); experts=np.zeros((n,3)); lower=np.full(n,cfg.epsilon/n)
    qrisk=policy_distribution('risk_only',benefit,risk,dummy,experts,lower,cfg)
    assert int(np.argmax(qrisk)) == int(np.argmax(risk)) != int(np.argmax(benefit))
    # Gaussian contrast noise: use identical noise in the two loss terms.
    x=rng.uniform(size=(20000,30)); b=rng.uniform(size=x.shape); m=rng.uniform(size=x.shape)
    noise=rng.normal(0,.05,x.shape)
    err=(((x+noise-b)**2-(x+noise-m)**2)-((x-b)**2-(x-m)**2)).mean(1)
    assert abs(err.mean())<5*err.std()/np.sqrt(len(err))
    return dict(max_solver_objective_gap=max(solver), max_unbiasedness_error=max(unbiased),
                max_gradient_error=max(gradient), min_variance_hessian_eigenvalue=min(psd),
                fixed_pool_ppat_trials=12000, native_ppat_mean_errors=mean_error.tolist(),
                gaussian_noise_cancellation_error=float(err.mean()), max_envelope_bound_ratio=max(envelopes),
                max_hard_certificate_ratio=max(certificates), max_quadratic_identity_error=max(quadratics),
                risk_only_regression_fixed=True,
                max_dual_primary_ratio=max(x[0] for x in dual_ratios),
                max_dual_write_ratio=max(x[1] for x in dual_ratios),
                max_dual_reference_mix=max(x[2] for x in dual_ratios))


def information_boundary_checks(memory=None, auditor=None):
    cfg=Geometry(height=64,width=96,fovea=16,block=8,factor=4,frames=10)
    acfg=MethodConfig(); method=Method('PEM_R9_TwinAudit','dual_joint','online',envelope=False,certificate=True,twin_audit=True)
    frames=synthetic_frames(cfg); obs=observations(frames,cfg,'boundary','nominal',0)
    a=run_sequence(frames,obs,cfg,acfg,'boundary','nominal',0,method,memory,auditor,keep=True)
    changed=frames.copy(); nx=cfg.width//cfg.fovea;s=cfg.fovea
    for t,row in a[1].iterrows():
        act=int(row.action); yy,xx=act//nx*s,act%nx*s
        mask=np.ones((cfg.height,cfg.width),bool);mask[yy:yy+s,xx:xx+s]=False
        changed[t][mask]=1-changed[t][mask]
    b=run_sequence(changed,obs,cfg,acfg,'boundary','nominal',0,method,memory,auditor,keep=True)
    keys=['action','propensity','augmented','write_augmented','weight_zero','weight_energy','weight_learned']
    assert np.allclose(a[1][keys].values,b[1][keys].values,equal_nan=True)
    assert np.array_equal(a[4]['reconstruction'],b[4]['reconstruction'])
    assert abs(a[0]['mse']-b[0]['mse'])>1e-5
    future=frames.copy();future[5:]=1-future[5:]
    future_obs=observations(future,cfg,'boundary','nominal',0)
    c=run_sequence(future,future_obs,cfg,acfg,'boundary','nominal',0,method,memory,auditor)
    assert np.allclose(a[1][keys].values[:5],c[1][keys].values[:5],equal_nan=True)
    for m in core_methods():
        _,t,_,_,_=run_sequence(frames,obs,cfg,acfg,'boundary','nominal',0,m,memory,auditor)
        assert np.isfinite(t[['augmented','mse','ssim','cs_low','cs_high']]).all().all()
        if m.twin_audit and t.write_valid.any():
            assert np.isfinite(t.loc[t.write_valid,['write_augmented','exact_variance_write']]).all().all()
        if m.policy=='uniform': assert np.allclose(t.naive,t.ipw)
        if m.policy=='ppat_2026_m1': assert np.allclose(t.augmented,t.ppat_m1)
    return dict(unqueried_pixels_invariant=True, future_frames_invariant=True,
                predictable_weight_timing=True, all_core_methods_executed=True,
                uniform_ipw_identity=True, native_one_query_ppat_identity=True,
                twin_state_counterfactual_executed=True, dual_audit_budget_checked=True)


def coverage_simulation(trials=400, frames=80):
    """Adaptive synthetic process; validates code, not natural-video performance."""
    rng=np.random.default_rng(1024); good=[]; final=[]
    for _ in range(trials):
        cs=NormalMixtureCS(.05,.01); truth=0.; ok=True
        for t in range(frames):
            # Bounded, time-varying predictable mean plus Gaussian noise.
            mean=.1*np.sin(t/5)+.04*np.tanh(cs.total)
            variance=.03**2*(1+abs(mean))
            y=mean+rng.normal(0,np.sqrt(variance));truth+=mean
            lo,hi,_=cs.update(y,variance)
            ok=ok and lo<=truth/(t+1)<=hi
        good.append(ok);final.append(hi-lo)
    rate=float(np.mean(good))
    assert rate>.90, 'Large coverage deficit in CS implementation simulation'
    return dict(trials=trials, frames=frames, simultaneous_coverage=rate,
                mean_final_width=float(np.mean(final)), nominal_coverage=.95,
                scope='synthetic adaptive Gaussian software diagnostic')


def baseline_interface_checks():
    """Torch-free checks of the SR/VSR interface: MATLAB-bicubic preview and
    strict causality of the windowed video-SR wrapper."""
    from video_core import matlab_bicubic_down, down
    from r9_neural import TemporalWindowSR
    const=np.full((64,112,3),.37,np.float32)
    assert np.allclose(matlab_bicubic_down(const,4),.37,atol=1e-6)
    ramp=np.tile(np.linspace(0,1,448,dtype=np.float32)[None,:,None],(64,1,3))
    assert np.allclose(matlab_bicubic_down(ramp,4)[:,2:-2],down(ramp,4)[:,2:-2],atol=1e-5)
    class Fake(TemporalWindowSR):
        def __init__(self,mode):
            self.mode=mode;self.causal=mode=='causal';self.window=3;self.device='cpu'
        def sync(self):pass
        def _run(self,stack):
            c=np.cumsum(stack,0)[::-1].cumsum(0)[::-1]  # depends on past AND future within the input
            return np.repeat(np.repeat(c,4,1),4,2)
    rng=np.random.default_rng(5)
    a=[rng.uniform(size=(8,8,3)).astype(np.float32) for _ in range(9)]
    b=a[:5]+[1-x for x in a[5:]]
    pa,_=Fake('causal').predict_sequence(a); pb,_=Fake('causal').predict_sequence(b)
    assert all(np.array_equal(x,y) for x,y in zip(pa[:5],pb[:5])), 'Causal VSR leaked future previews'
    oa,_=Fake('offline').predict_sequence(a); ob,_=Fake('offline').predict_sequence(b)
    assert not np.array_equal(oa[0],ob[0]), 'Offline reference should read the whole clip'
    return dict(matlab_bicubic_constant=True,matlab_bicubic_ramp=True,causal_vsr_future_invariant=True,
                offline_vsr_flagged_noncausal=True)
'''
(SOURCE_DIR / 'r9_checks.py').write_text(SOURCE, encoding='utf-8')
print('Wrote', 'r9_checks.py')

## 10. Experiment runner

In [ ]:
SOURCE = r'''"""Experiment runner."""
from dataclasses import asdict, dataclass, replace
from pathlib import Path
import hashlib, importlib.metadata, json, os, platform, sys, time, zipfile
import cv2, joblib, numpy as np, pandas as pd
from r9_data import (DATASETS, Clip, _clip, sha256, digest_json, atomic_json, immutable_json,
    download, extract_dataset, discover_davis, discover_reds,
    validate_manifest, load_clip, synthetic_frames)
from r9_core import MethodConfig, AuditEnsemble
from r9_engine import (Geometry, Method, core_methods, neural_methods, observations, previews, run_sequence)
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, wait, FIRST_COMPLETED
from r9_neural import SPECS, load_baselines, baseline_sanity
from r9_stats import export_analysis, export_backbone_comparisons, publication_figures, trace_figures, paper_tables, CONTEXT
from video_core import RiskMemory


@dataclass
class RunConfig:
    root: str
    profile: str = 'smoke'  # smoke, laptop, paper
    experiment_id: str = 'r9_01'
    run_neural: bool = True
    device: str = 'auto'
    download_data: bool = True
    dataset_roots: dict = None
    threads: int = 2
    export_figures: bool = True
    line_dpi: int = 1000
    high_resolution_reds: bool = False
    save_all_traces: bool = False
    workers: int = 0               # CPU processes for the controller/evaluator; 0 = automatic
    data_root: str = None          # downloads/extractions/synthetic smoke data; defaults to root.
    include_video_sr: bool = True  # BasicVSR++ (CVPR 2022) causal + offline reference
    neural_workers: int = 4        # processes computing neural baseline predictions (speed only)
    neural_threads: int = 2        # PyTorch threads per neural worker (speed only)


def environment():
    packages = ['numpy','scipy','pandas','scikit-learn','matplotlib','opencv-python-headless',
                'torch','einops','joblib','psutil']
    versions = {}
    for name in packages:
        try: versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: versions[name] = 'not installed'
    result = dict(python=sys.version, platform=platform.platform(), packages=versions)
    try:
        import torch
        result.update(cuda=torch.version.cuda, cuda_available=torch.cuda.is_available(),
            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            torch_threads=torch.get_num_threads())
    except ImportError: pass
    return result


def _stable_environment(v):
    """Only package versions enter the lock; host/GPU names may change between
    sessions without changing the experiment."""
    return dict(python=v['python'].split()[0],packages=v['packages'])


def source_hashes():
    return {p.name:sha256(p) for p in sorted(Path(__file__).parent.glob('*.py'))}


def _choose(clips, n):
    ordered = sorted(clips, key=lambda c: hashlib.sha256((c.dataset+'/'+c.clip_id).encode()).hexdigest())
    return ordered if n is None else ordered[:n]


def prepare_data(config, split):
    root=Path(config.data_root or config.root); profile=config.profile
    if profile not in ['smoke','laptop','paper']: raise ValueError('Unknown profile')
    if profile=='smoke':
        geo=Geometry(height=64,width=96,fovea=16,frames=12)
        clips=[]
        for role,count,offset in [('fit',4,100),('development',2,200),('external_test',4,300)]:
            for i in range(count):
                name=f'{role}_{i:02d}'; folder=root/'data'/'synthetic'/name
                folder.mkdir(parents=True,exist_ok=True)
                for t,frame in enumerate(synthetic_frames(geo,offset+i,cut=i%2==1)):
                    path=folder/f'{t:04d}.png'
                    pixels=np.rint(frame[...,::-1]*255).astype(np.uint8)
                    # Regenerate the test frames.
                    if not cv2.imwrite(str(path),pixels): raise IOError(path)
                clips.append(_clip('Synthetic',name,name,role,sorted(folder.glob('*.png'))))
        clips,report=validate_manifest(clips)
        return clips,geo,report
    geo=Geometry(frames=100 if profile=='paper' else 32)
    roots=config.dataset_roots or {}
    resolved={}
    for name,entry in DATASETS.items():
        if name in roots:
            p=Path(roots[name]).expanduser().resolve()
            if not p.is_dir(): raise FileNotFoundError(p)
            resolved[name]=p
            continue
        if not config.download_data or not entry.get('url'):
            raise FileNotFoundError(f'Set dataset_roots[{name!r}] to the local {name} folder ({entry["source"]}). No substitute data is used.')
        print('Preparing official dataset:',name,flush=True)
        filename={'DAVIS':'DAVIS-2017-trainval-480p.zip','REDS':'val_sharp.zip'}[name]
        try:
            archive=download(entry['url'],root/'downloads'/filename,entry.get('sha256'))
            resolved[name]=extract_dataset(archive,root/'data'/name,name)
        except Exception as error:
            raise RuntimeError(f'{name} download failed. Obtain the dataset from {entry["source"]}. Completed work is retained; no substitute data will be silently used.') from error
    davis=discover_davis(resolved['DAVIS'],split,geo.frames)
    if profile=='laptop':
        davis=sum([_choose([c for c in davis if c.role==role],n)
                   for role,n in [('fit',12),('development',4),('legacy_test',10)]],[])
    # Both REDS splits must be complete on disk; the laptop pilot then uses the first 10 clips of each.
    limit=None if profile=='paper' else 10
    reds=discover_reds(resolved['REDS'],geo.frames,limit,dataset='REDS',expected=30)
    reds_train=discover_reds(resolved['REDS_train'],geo.frames,limit,dataset='REDS_train',expected=240)
    if profile=='paper':
        if len(reds_train)!=240: raise ValueError('Paper profile requires all 240 REDS train_sharp clips')
        if len(reds)!=30: raise ValueError('Paper profile requires all 30 REDS validation clips')
        if len(davis)!=90: raise ValueError('Legacy DAVIS manifest must retain all 90 original sources')
        if any(len(c.paths)!=100 for c in reds+reds_train): raise ValueError('REDS paper clips must have 100 frames')
    clips,report=validate_manifest(davis+reds+reds_train)
    report['dataset_roots']={k:str(v) for k,v in resolved.items()}
    report['pilot_scope']=('laptop pilot: DAVIS (12 fit, 4 development, 10 legacy test) + first 10 REDS val + first 10 REDS train clips, 32 frames'
                           if profile=='laptop' else 'all configured datasets')
    report['official_sources']=DATASETS
    report['reds_train_role']='REDS train_sharp replaces Vimeo-90K and is used only as unseen external test data: no model in this study is trained on REDS.'
    report['limitation']='Content hashing detects exact duplicates; it cannot certify absence of near-duplicates or pretrained data overlap.'
    report['content_digests']=dataset_digests(clips)
    report['clip_counts']={d:sum(c.dataset==d for c in clips) for d in sorted({c.dataset for c in clips})}
    return clips,geo,report

def dataset_digests(clips):
    """Order-independent content digest per dataset; published so readers can verify
    that the exact same frames (including the unpinned REDS mirror) were evaluated."""
    out={}
    for d in sorted({c.dataset for c in clips}):
        items=sorted(c.clip_id+':'+c.content_sha256 for c in clips if c.dataset==d)
        out[d]=hashlib.sha256('\n'.join(items).encode()).hexdigest()
    return out


def _lockable_clip(c):
    """Clip identity by content, not by filesystem location, so a session that
    re-extracts data to another disk resumes the same locked experiment."""
    d=asdict(c); d.pop('paths'); return d


def fit_models(clips,geo,acfg,out,profile):
    fit=sorted([c for c in clips if c.role=='fit'],key=lambda c:c.clip_id)
    if len(fit)<4: raise ValueError('At least four fitting sources are required')
    cut=len(fit)//2; memory_sources=fit[:cut]; auditor_sources=fit[cut:]
    provenance=dict(memory_sources=[c.source_group for c in memory_sources],
                    auditor_sources=[c.source_group for c in auditor_sources],
                    labels='only selected, noisy foveal observations; no dense clean targets',
                    harvest_policy='uniform',regimes=['nominal','noisy'],
                    memory_auditor_source_disjoint=True, group_oof_mean_folds=min(5,len(auditor_sources)))
    assert not set(provenance['memory_sources'])&set(provenance['auditor_sources'])
    path=out/'models.joblib'; meta=out/'training_provenance.json'
    if path.exists() and meta.exists():
        saved=json.loads(meta.read_text())
        if saved.get('model_sha256')!=sha256(path): raise ValueError('Fitted-model checksum mismatch')
        # Load only this run's locally generated model; never an uploaded pickle.
        return (*joblib.load(path),saved)
    memory_rows=[]
    harvest=Method('training_uniform','uniform','zero',envelope=False)
    for index,c in enumerate(memory_sources):
        frames=load_clip(c,geo)
        for regime in ['nominal','noisy']:
            obs=observations(frames,geo,c.source_group,regime,0)
            _,_,_,mr,_=run_sequence(frames,obs,geo,acfg,c.source_group,regime,0,harvest,collect=True)
            memory_rows.extend(mr)
        print(f'Memory fit harvest {index+1}/{len(memory_sources)}',flush=True)
    X,C,H,R=[np.concatenate([row[k] for row in memory_rows]) for k in range(4)]
    memory=RiskMemory(leaf=4 if profile=='smoke' else 40).fit(X,C,H,R)
    rows=[]
    for index,c in enumerate(auditor_sources):
        frames=load_clip(c,geo)
        for regime in ['nominal','noisy']:
            obs=observations(frames,geo,c.source_group,regime,0)
            _,_,tr,_,_=run_sequence(frames,obs,geo,acfg,c.source_group,regime,0,harvest,memory,collect=True)
            rows.extend(tr)
        print(f'Auditor fit harvest {index+1}/{len(auditor_sources)}',flush=True)
    Z=np.stack([r[0] for r in rows]); d=np.array([r[1] for r in rows]); e=np.array([r[2] for r in rows]); groups=np.array([r[3] for r in rows])
    auditor=AuditEnsemble(trees=16 if profile=='smoke' else 64,leaf=4 if profile=='smoke' else 12).fit(Z,d,e,groups)
    joblib.dump((memory,auditor),path,compress=3)
    provenance.update(memory_training_blocks=len(X),auditor_observations=len(Z),
                      source_oof_proxy_mse=auditor.oof_mse,model_sha256=sha256(path))
    atomic_json(meta,provenance)
    return memory,auditor,provenance


_WORKER = {}


def _worker_init(model_path):
    """Runs once per spawned worker. 'spawn' (not fork) keeps CUDA state out of
    the workers, so GPU baselines in the parent process are safe."""
    for name in ['OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS']:
        os.environ[name] = '1'
    cv2.setNumThreads(1)
    memory, auditor = joblib.load(model_path)
    _WORKER.update(memory=memory, auditor=auditor)


def auto_workers(requested=0):
    if requested and requested > 0:
        return int(requested)
    cpus = os.cpu_count() or 1
    try:
        import psutil
        gb = psutil.virtual_memory().available / 2 ** 30
        by_memory = max(1, int((gb - 2) // 1.5))
    except Exception:
        by_memory = cpus
    return max(1, min(cpus, by_memory, 12))


def make_pool(workers, model_path):
    for name in ['OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS']:
        os.environ[name] = '1'
    return ProcessPoolExecutor(max_workers=workers, mp_context=multiprocessing.get_context('spawn'),
                               initializer=_worker_init, initargs=(str(model_path),))


def _neural_worker_init(model_path, official_root, device, include_video, threads, lock):
    """Neural worker: the CPU-worker state plus its own copy of the frozen baselines.
    Model loading (which verifies checksums and rewrites provenance files) is
    serialized with a lock so workers never write the same file concurrently."""
    _worker_init(model_path)
    import torch
    torch.set_num_threads(threads)
    torch.manual_seed(20260922)
    torch.use_deterministic_algorithms(True, warn_only=True)
    with lock:
        _WORKER['neural'] = load_baselines(official_root, device, include_video)


def make_neural_pool(workers, model_path, official_root, device, include_video, threads):
    for name in ['OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS']:
        os.environ[name] = '1'
    ctx = multiprocessing.get_context('spawn')
    return ProcessPoolExecutor(max_workers=workers, mp_context=ctx, initializer=_neural_worker_init,
                               initargs=(str(model_path), str(official_root), device, include_video, threads, ctx.Lock()))


def _neural_execute_job(payload):
    """Same computation as the serial path: _neural_predictions, then _execute_job,
    but inside a worker so several neural jobs run at once. Predictions never leave
    the worker, so no large arrays are pickled between processes."""
    preds, times = _neural_predictions(payload['job'], _WORKER['neural'])
    return _execute_job(dict(payload, predictions=preds, neural_times=times))


def _dev_task(args):
    c, geo, acfg, candidates, regime = args
    memory, auditor = _WORKER['memory'], _WORKER['auditor']
    frames = load_clip(c, geo)
    obs = observations(frames, geo, c.source_group, regime, 0)
    methods = [(-1, acfg, Method('greedy', 'greedy', 'learned', envelope=False, adapt_moment=False,
                                 adapt_weights=False, certificate=False))]
    methods += [(i, a, Method('candidate', 'dual_joint', 'online', envelope=False, certificate=True, twin_audit=True))
                for i, a in enumerate(candidates)]
    rows = []
    for i, a, m in methods:
        s, *_ = run_sequence(frames, obs, geo, a, c.source_group, regime, 0, m, memory, auditor)
        rows.append(dict(candidate=i, source=c.source_group, regime=regime, mse=s['mse'],
                         pqv=s['augmented_pqv'], write_pqv=s['write_pqv']))
    return rows


def _execute_job(payload):
    """Run every method of one (clip, regime, draw, geometry) job and checkpoint it.
    Common random numbers: all methods share the preview noise, patch noise and
    gaze uniforms, so method differences are paired within a job."""
    study, c, regime, draw, g, a, ms = payload['job']
    memory, auditor = _WORKER['memory'], _WORKER['auditor']
    frames = load_clip(c, g)
    obs = observations(frames, g, c.source_group, regime, draw)
    predictions = payload.get('predictions') or {}
    neural_times = payload.get('neural_times') or {}
    rows, kept = [], None
    for method in ms:
        keep = payload['keep'] and method.name == 'PEM_R9_TwinAudit'
        s, trace, _, _, example = run_sequence(frames, obs, g, a, c.source_group, regime, draw, method, memory, auditor,
                                               predictions.get(method.neural), neural_times.get(method.neural), keep=keep)
        s.update(dataset=c.dataset, clip_id=c.clip_id, source_group=c.source_group, role=c.role,
                 regime=regime, draw=draw, study=study, method=method.name,
                 height=g.height, width=g.width, factor=g.factor, fovea=g.fovea,
                 preview_kernel=g.preview_kernel, kappa=a.kappa, write_kappa=a.write_kappa,
                 certificate_kappa=a.certificate_kappa)
        rows.append(s)
        if payload.get('trace_dir'):
            Path(payload['trace_dir']).mkdir(exist_ok=True)
            trace.to_csv(Path(payload['trace_dir']) / (payload['key'] + '_' + method.name + '.csv.gz'),
                         index=False, compression='gzip')
        if keep:
            kept = (trace, example)
    atomic_json(payload['checkpoint'], dict(job_hash=payload['key'], rows=rows))
    return payload['index'], rows, kept


def select_on_development(clips,geo,acfg,out,profile,workers=1):
    path=out/'development_selection.json'
    if path.exists():
        report=json.loads(path.read_text());return MethodConfig(**report['selected']),report
    candidates=[replace(acfg,learning_rate=lr,certificate_kappa=ck,write_kappa=wk)
                for lr in ([.05,.15] if profile=='smoke' else [.05,.15,.4])
                for ck in ([1.5] if profile=='smoke' else [1.15,1.5,2.5])
                for wk in ([2.0] if profile=='smoke' else [1.25,2.0,4.0])]
    dev=[c for c in clips if c.role=='development']
    tasks=[(c,geo,acfg,candidates,regime) for c in dev for regime in ['nominal','noisy']]
    rows=[]
    with make_pool(min(workers,len(tasks)),out/'models.joblib') as pool:
        for k,part in enumerate(pool.map(_dev_task,tasks)):
            rows.extend(part)
            print(f'Development selection {k+1}/{len(tasks)}',flush=True)
    df=pd.DataFrame(rows); grouped=df.groupby('candidate')[['mse','pqv','write_pqv']].mean()
    grouped['dual_audit_score']=np.sqrt(np.maximum(grouped.pqv,1e-30)*np.maximum(grouped.write_pqv,1e-30))
    threshold=1.05*grouped.loc[-1,'mse'];eligible=[i for i in range(len(candidates)) if grouped.loc[i,'mse']<=threshold]
    fallback=not eligible
    selected=min(eligible,key=lambda i:(grouped.loc[i,'dual_audit_score'],i)) if eligible else min(range(len(candidates)),key=lambda i:(grouped.loc[i,'mse'],grouped.loc[i,'dual_audit_score'],i))
    report=dict(selected=asdict(candidates[selected]),selected_candidate=selected,
                candidates=[asdict(c) for c in candidates], reconstruction_threshold=float(threshold),
                rule='Lowest geometric mean of primary-audit PQV and twin-write PQV among candidates with MSE <= 1.05 x greedy development MSE; otherwise lowest MSE, then dual-audit score, then candidate index.',
                fallback_to_reconstruction=fallback,development_sources=sorted(df.source.unique()),
                candidate_scores=grouped.reset_index().to_dict('records'))
    df.to_csv(out/'development_scores.csv',index=False);immutable_json(path,report)
    return candidates[selected],report


def _named(method, name):
    return replace(method, name=name)


def make_jobs(clips,geo,acfg,config):
    """Prespecified evaluation plan.

    Study roles (paper profile):
      main          Long natural videos (REDS val 100 frames, DAVIS legacy 480p) -
                    the primary evidence for temporal memory and auditing.
      reds_train_full   All 240 REDS train_sharp clips (100 frames, unseen by
                    every model) - scale / generalisation evidence.
      reds_train_detail Fixed 60-clip REDS train subset with every controlled policy.
      neural_*      Published SR / video-SR backbones in the identical sensor
                    interface, under the checkpoints' native degradation
                    (clean MATLAB bicubic) and under the paper's sensor model.
      timing        Serial (uncontended) latency measurement.
      sensitivity   kappa, certificate, write-kappa, preview factor/kernel,
                    and a deliberately assumption-violating negative control.
    """
    test=[c for c in clips if c.role in ['external_test','legacy_test']]
    profile=config.profile
    m=core_methods(); by={x.name:x for x in m}
    ours=by['PEM_R9_TwinAudit']
    # Internal predecessor 'R7 reserve' is omitted from reporting studies; it is
    # superseded by the certified variant and adds no information for readers.
    # PEM_R9_single_audit is definitionally identical to PEM_R8_certified (same
    # policy, proxy and certificate; the twin audit is only measured), so it is
    # not reported twice.
    controlled=[x for x in m if x.name not in ('PEM_R7_reserve','PEM_R9_single_audit')]
    jobs=[]
    causal=list(SPECS)+(['BasicVSRpp2022_causal'] if config.include_video_sr else [])
    offline=['BasicVSRpp2022_offline'] if config.include_video_sr else []
    neural_list=[by['PEM_uniform'],by['PEM_R6_fixed'],ours]+neural_methods(causal,offline)

    def add(study,cs,regimes,gs,ms,draws_,ac=acfg):
        for c in cs:
            for regime in regimes:
                for draw in draws_:
                    jobs.append((study,c,regime,draw,gs,ac,ms))

    def sensitivity_block(sens,draws_):
        for k in [1.,2.,4.,8.]:
            add('kappa',sens,['nominal'],geo,[_named(ours,f'PEM_R9_k{k:g}')],draws_,replace(acfg,kappa=k))
        for ck in [1.05,1.25,1.5,2.,4.]:
            add('certificate',sens,['nominal'],geo,[_named(ours,f'PEM_R9_cert{ck:g}')],draws_,replace(acfg,certificate_kappa=ck))
        for wk in [1.,1.25,2.,4.,8.]:
            add('write_kappa',sens,['nominal'],geo,[_named(ours,f'PEM_R9_writek{wk:g}')],draws_,replace(acfg,write_kappa=wk))
        transfer=[by['PEM_uniform'],by['PEM_greedy'],by['PEM_R6_fixed'],by['PEM_R8_certified'],ours]
        add('scale8',sens,['nominal'],replace(geo,factor=8),transfer,draws_)
        add('bicubic',sens,['nominal'],replace(geo,preview_kernel='bicubic'),transfer,draws_)
        add('bias_negative_control',sens,['biased_stress'],geo,[ours],draws_)

    if profile=='paper':
        reds_train=[c for c in test if c.dataset=='REDS_train']
        reds=[c for c in test if c.dataset=='REDS']
        davis=[c for c in test if c.dataset=='DAVIS']
        long_clips=reds+davis
        # 1. Primary evidence: long clips, both sensor regimes, two randomisation draws.
        add('main',long_clips,['nominal','drift'],geo,controlled,range(2))
        # Secondary variant (post-pilot amendment): TwinAudit with write_kappa=1. Not the primary
        # method; TwinAudit at the development-selected write_kappa stays the target of all comparisons.
        add('main',long_clips,['nominal','drift'],geo,[_named(ours,'PEM_R9_TwinAudit_strict')],range(2),replace(acfg,write_kappa=1.))
        # 2. Scale: all 240 REDS train clips (unseen by every model).
        add('reds_train_full',reds_train,['nominal'],geo,
            [by['PEM_greedy'],by['PEM_R6_fixed'],by['PPAT2026_M1'],ours],range(1))
        # 3. All controlled policies on a fixed REDS train subset.
        add('reds_train_detail',_choose(reds_train,60),['nominal'],geo,controlled,range(2))
        if config.run_neural:
            nset=reds+_choose(davis,10)+_choose(reds_train,20)
            add('neural_native',nset,['clean'],replace(geo,preview_kernel='bicubic'),neural_list,range(1))
            add('neural_sensor',nset,['nominal'],geo,neural_list,range(1))
            add('timing',_choose(reds,10),['nominal'],geo,
                [by['PEM_greedy'],by['PEM_R6_fixed'],ours]+neural_methods(causal[-1:],offline),range(1))
        else:
            add('timing',_choose(reds,10),['nominal'],geo,[by['PEM_greedy'],by['PEM_R6_fixed'],ours],range(1))
        sensitivity_block(_choose(reds,15)+_choose(davis,15)+_choose(reds_train,15),range(1))
        if config.high_resolution_reds:
            add('high_resolution',reds,['nominal'],replace(geo,height=448,width=768),
                [by['PEM_uniform'],by['PEM_greedy'],by['PEM_R6_fixed'],ours],range(1))
        return jobs

    draws=range(1 if profile=='smoke' else 2)
    add('main',test,['nominal','drift'],geo,controlled,draws)
    # Secondary variant (post-pilot amendment): TwinAudit with write_kappa=1. Not the primary
    # method; TwinAudit at the development-selected write_kappa stays the target of all comparisons.
    add('main',test,['nominal','drift'],geo,[_named(ours,'PEM_R9_TwinAudit_strict')],draws,replace(acfg,write_kappa=1.))
    per=1 if profile=='smoke' else 12
    subset=[]
    for ds in sorted({c.dataset for c in test}):
        subset.extend(_choose([c for c in test if c.dataset==ds],per))
    if config.run_neural:
        add('neural_native',subset,['clean'],replace(geo,preview_kernel='bicubic'),neural_list,range(1))
        add('neural_sensor',subset,['nominal'],geo,neural_list,range(1))
    add('timing',subset[:max(1,min(len(subset),4))],['nominal'],geo,[by['PEM_greedy'],by['PEM_R6_fixed'],ours],range(1))
    sensitivity_block(subset,draws)
    return jobs

def coverage_table(df,out,reps):
    rng=np.random.default_rng(3519);rows=[]
    keys=CONTEXT+['method']
    for context,g in df.groupby(keys):
        a=g.groupby('source_group')[['cs_simultaneous_covered','cs_final_covered','cs_final_width']].mean()
        row=dict(zip(keys,context));row['source_clusters']=len(a)
        for metric in a:
            values=a[metric].to_numpy();boots=[]
            for start in range(0,reps,256):
                ids=rng.integers(0,len(a),(min(256,reps-start),len(a)))
                boots.extend(values[ids].mean(1))
            row[metric]=float(values.mean());row[metric+'_ci_low'],row[metric+'_ci_high']=np.quantile(boots,[.025,.975])
        row['assumptions_hold']=bool(g.cs_valid_assumptions.all())
        row['interval_limitation']='Percentile source bootstrap can collapse when all repetitions have identical coverage; this is not proof of nominal coverage.'
        rows.append(row)
    pd.DataFrame(rows).to_csv(out/'coverage_summary.csv',index=False)


def _job_entry(job):
    study,c,regime,draw,g,a,ms=job
    return dict(study=study,dataset=c.dataset,clip_id=c.clip_id,source_group=c.source_group,
                role=c.role,content_sha256=c.content_sha256,regime=regime,draw=draw,
                geometry=asdict(g),audit_config=asdict(a),methods=[asdict(m) for m in ms])


def _seed_torch(*parts):
    import torch
    from video_core import seed
    torch.manual_seed(seed(*parts))


def _neural_predictions(job,neural):
    study,c,regime,draw,g,a,ms=job
    names=sorted({m.neural for m in ms if m.neural})
    if not names: return {},{}
    frames=load_clip(c,g); pv=previews(frames,g,c.source_group,regime,draw)
    preds,times={},{}
    for name in names:
        # The official SCMSR code samples (gumbel_softmax) even at inference, so its output
        # depends on the global PyTorch RNG. Seed it per clip, condition and model so predictions
        # do not depend on the process, job order, parallel scheduling or resume history.
        _seed_torch(g.seed,c.dataset,c.clip_id,regime,draw,g.preview_kernel,g.factor,name)
        pp,tt=neural[name].predict_sequence(pv); preds[name]=list(pp); times[name]=list(tt)
    return preds,times


def run_pipeline(config,split):
    config.root=str(Path(config.root).expanduser().resolve());root=Path(config.root)
    root.mkdir(parents=True,exist_ok=True);cv2.setNumThreads(config.threads)
    if config.data_root: config.data_root=str(Path(config.data_root).expanduser().resolve())
    workers=auto_workers(config.workers)
    if config.run_neural:
        os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
        try:
            import torch
        except ImportError:
            if config.profile!='smoke': raise
            torch=None
    if config.run_neural and torch is not None:
        torch.set_num_threads(config.threads)
        torch.manual_seed(20260922)
        # warn_only: inference-only ops are deterministic; this avoids a hard
        # failure on kernels that merely lack a deterministic *backward*.
        torch.use_deterministic_algorithms(True,warn_only=True)
        if torch.cuda.is_available():
            torch.backends.cudnn.benchmark=False
            torch.backends.cudnn.deterministic=True
            torch.backends.cuda.matmul.allow_tf32=False
            torch.backends.cudnn.allow_tf32=False
            torch.cuda.reset_peak_memory_stats()
    start=time.perf_counter();clips,geo,data_report=prepare_data(config,split)
    versions=environment()
    lock_config={k:v for k,v in asdict(config).items() if k not in ['root','data_root','dataset_roots','threads','device','download_data','export_figures','line_dpi','workers','save_all_traces','neural_workers','neural_threads']}
    base_lock=dict(config=lock_config,geometry=asdict(geo),method_defaults=asdict(MethodConfig()),
        clips=[_lockable_clip(c) for c in clips],code=source_hashes(),environment=_stable_environment(versions),
        official_specs=SPECS if config.run_neural else {},legacy_splits=split)
    fingerprint=digest_json(base_lock)
    out=root/'results'/(config.experiment_id+'_'+fingerprint[:12]);out.mkdir(parents=True,exist_ok=True)
    immutable_json(out/'inputs.lock.json',base_lock);atomic_json(out/'data_integrity.json',data_report)
    print('Experiment:',out.name,'| sources:',len({c.source_group for c in clips}),'|',config.profile,'| CPU workers:',workers,flush=True)
    memory,auditor,training=fit_models(clips,geo,MethodConfig(),out,config.profile)
    acfg,selection=select_on_development(clips,geo,MethodConfig(),out,config.profile,workers)
    neural={}; sanity={}
    if config.run_neural:
        neural=load_baselines(root/'official_models',config.device,config.include_video_sr)
        if config.profile!='smoke':
            dev=sorted([c for c in clips if c.role=='development'],key=lambda c:c.clip_id)[:2]
            sanity=baseline_sanity(neural,[load_clip(c,replace(geo,frames=10)) for c in dev],geo.factor)
            atomic_json(out/'baseline_sanity.json',sanity)
            print('Baseline sanity gate passed:',{k:round(v['mean_psnr_gain_over_bicubic_db'],2) for k,v in sanity.items()},flush=True)
    jobs=make_jobs(clips,geo,acfg,config)
    job_entries=[_job_entry(j) for j in jobs]
    locked=dict(input_fingerprint=fingerprint,training=training,selection=selection,
                official_models={n:m.provenance for n,m in neural.items()},jobs=job_entries,
                expected_jobs=len(jobs),expected_rows=sum(len(j[-1]) for j in jobs),
                inference='Paired source-cluster resampling; equal source weights; primary endpoints reconstruction MSE, memory-benefit audit PQV, and one-step write-value audit PQV.',
                historical_disclosure='The 40 legacy DAVIS test sources were seen in earlier revisions. REDS train_sharp and REDS val_sharp were never used for design, fitting or selection; no model in this study is trained on REDS (PEM models fit on DAVIS fit sources; BasicVSR++ checkpoint trained on the Vimeo-90K training split; NTIRE 2025 ESR checkpoints are still-image models).',
                baseline_disclosure='NTIRE 2025 ESR checkpoints and the BasicVSR++ Vimeo-90K BI checkpoint are external author weights evaluated without fine-tuning. The BasicVSR++ offline variant reads future previews and is a non-causal reference only.',
                negative_control='biased_stress intentionally violates zero-mean noise; no CS guarantee applies.',
                post_pilot_amendment='Added after the laptop pilot (local_01_677c295da848) and before the paper run: paired same-backbone comparisons in the neural studies, backbone + TwinAudit gaze (_memory_R9) vs backbone + memory (_memory) and vs backbone + patch (_patch), for PSNR, MSE, audit PQV and write PQV, using the same source-cluster bootstrap, sign-flip test and Holm adjustment (within study and metric). Also added at the same time: a secondary variant PEM_R9_TwinAudit_strict (TwinAudit with write_kappa = 1, displayed as "TwinAudit-strict (κw = 1)") in the main study only. It is a secondary variant, NOT the primary method: TwinAudit with the development-selected write_kappa = 2 remains the primary method and the target of all paired comparisons, and the development selection was not changed. Speed-only changes: THREADS raised from 2 to 8, and neural baseline predictions computed in parallel worker processes (each job computed by the same functions as before; equivalence verified before the paper run). Reproducibility fix: the official SCMSR code samples gumbel_softmax even at inference, so the global PyTorch RNG is now seeded per clip, condition and model before each baseline prediction (and per model in the sanity gate); SCMSR numbers therefore differ slightly from the laptop pilot, where its random draws depended on call order. No method, selection rule or existing statistic was changed.')
    immutable_json(out/'protocol.lock.json',locked)
    plan={}
    for job in jobs:
        plan[job[0]]=plan.get(job[0],0)+len(job[-1])
    print('Planned method runs:',plan,flush=True)
    frames_planned=sum(len(j[-1])*min(len(j[1].paths),j[4].frames) for j in jobs)
    print(f'Planned frame-method evaluations: {frames_planned:,}',flush=True)
    checkpoint=out/'checkpoints';checkpoint.mkdir(exist_ok=True)
    trace_dir=str(out/'traces') if config.save_all_traces else None
    keep_index=next((i for i,j in enumerate(jobs) if j[0]=='main' and j[2]=='nominal' and j[3]==0
                     and any(m.name=='PEM_R9_TwinAudit' for m in j[-1])),None)
    if (out/'example.npz').exists(): keep_index=None
    results={};pending=[]
    for index,(job,entry) in enumerate(zip(jobs,job_entries)):
        key=digest_json(dict(job=entry,input=fingerprint,training=training['model_sha256'],selection=asdict(acfg)))
        cp=checkpoint/(key+'.json')
        if cp.exists():
            saved=json.loads(cp.read_text())
            if saved['job_hash']!=key or len(saved['rows'])!=len(job[-1]):raise ValueError('Invalid checkpoint')
            results[index]=saved['rows']
        else:
            pending.append(dict(index=index,job=job,key=key,checkpoint=str(cp),keep=index==keep_index,trace_dir=trace_dir))
    print(f'Locked {len(jobs)} jobs / {locked["expected_rows"]} method-sequence runs; {len(results)} jobs already complete. Restarting resumes.',flush=True)
    kept=None;session_start=time.perf_counter();done=0
    def record(res):
        nonlocal kept,done
        i,rows,k=res;results[i]=rows;done+=1
        if k is not None:
            trace,example=k;trace.to_csv(out/'example_trace.csv',index=False)
            np.savez_compressed(out/'example.npz',**example);atomic_json(out/'example_selection.json',job_entries[i])
            kept=k
        if done%25==0 or done==len(pending):
            el=time.perf_counter()-session_start
            print(f'Completed {len(results)}/{len(jobs)} jobs; session {el/3600:.2f} h; rough ETA {el/done*(len(pending)-done)/3600:.2f} h',flush=True)
    needs_neural=lambda p:any(m.neural for m in p['job'][-1])
    serial=[p for p in pending if p['job'][0]=='timing']
    neural_jobs=[p for p in pending if p['job'][0]!='timing' and needs_neural(p)]
    cpu_jobs=[p for p in pending if p['job'][0]!='timing' and not needs_neural(p)]
    if neural_jobs:
        # Neural baselines: each worker computes the SR predictions for one job and runs
        # that job, so several neural jobs proceed in parallel (speed only).
        nw=max(1,min(config.neural_workers,len(neural_jobs)))
        print(f'Neural jobs: {len(neural_jobs)} on {nw} workers x {config.neural_threads} PyTorch threads',flush=True)
        with make_neural_pool(nw,out/'models.joblib',root/'official_models',config.device,
                              config.include_video_sr,config.neural_threads) as pool:
            inflight={pool.submit(_neural_execute_job,p) for p in neural_jobs}
            while inflight:
                finished,inflight=wait(inflight,return_when=FIRST_COMPLETED)
                for f in finished: record(f.result())
    if cpu_jobs:
        with make_pool(workers,out/'models.joblib') as pool:
            inflight={pool.submit(_execute_job,p) for p in cpu_jobs}
            while inflight:
                finished,inflight=wait(inflight,return_when=FIRST_COMPLETED)
                for f in finished: record(f.result())
    if serial:
        # Latency study: executed alone in this process, with no competing workers.
        _WORKER.update(memory=memory,auditor=auditor)
        for p in serial:
            preds,times=_neural_predictions(p['job'],neural) if needs_neural(p) else ({},{})
            record(_execute_job(dict(p,predictions=preds,neural_times=times)))
    df=pd.DataFrame([row for i in range(len(jobs)) for row in results[i]])
    if len(df)!=locked['expected_rows']:raise AssertionError('Incomplete protocol')
    if df.duplicated(CONTEXT+['method','clip_id','draw']).any():raise AssertionError('Duplicate experiment row')
    required=['mse','ssim','augmented_pqv','augmented_squared_error','write_pqv','write_squared_error','online_total_ms_mean']
    if not np.isfinite(df[required].to_numpy()).all():raise AssertionError('Nonfinite scientific result')
    reps=1000 if config.profile=='smoke' else 6000
    summary,paired,controls=export_analysis(df,out,config.profile=='smoke',reps)
    coverage_table(df,out,reps)
    tables=paper_tables(summary,paired,out/'tables',neural_provenance={n:m.provenance for n,m in neural.items()})
    backbone=export_backbone_comparisons(df,out,config.profile=='smoke',reps)
    example_saved=(out/'example.npz').exists()
    if config.export_figures:
        publication_figures(df,summary,paired,controls,out/'figures',config.profile=='smoke',config.line_dpi)
        if example_saved:
            if kept is None:kept=(pd.read_csv(out/'example_trace.csv'),dict(np.load(out/'example.npz',allow_pickle=True)))
            trace_figures(*kept,out/'figures',config.profile=='smoke')
    import psutil
    runtime=dict(elapsed_seconds=time.perf_counter()-start,process_rss_at_end_bytes=psutil.Process().memory_info().rss,
                 cpu_workers=workers,
                 timing_scope="Report latency ONLY from study == 'timing', which runs serially without competing processes. Per-frame times in other studies were measured under parallel load. Times include preview/flow, controller/audit and SR with device transfers; evaluator, disk IO, fitting and downloads are excluded.",
                 reused_neural_predictions='SR predictions are computed once per common preview, but measured SR time is charged to every method using them.',
                 peak_cuda_allocated_bytes=None)
    if config.run_neural:
        try:
            import torch
            if torch.cuda.is_available():runtime['peak_cuda_allocated_bytes']=torch.cuda.max_memory_allocated()
        except ImportError: pass
    atomic_json(out/'runtime.json',runtime)
    complete_paper=config.profile=='paper' and config.run_neural
    readiness=dict(protocol_complete=True,profile=config.profile,rows=len(df),
        large_scale_protocol_executed=complete_paper,submission_ready=False,
        baselines_executed=sorted(neural),
        blockers=['Scientific superiority and practical gains must be read from the actual tables, including negative results.',
                  'Neural baselines are external checkpoints evaluated in a common one-patch sensor interface (native bicubic and sensor-noise conditions); they are not their original benchmark protocols.',
                  'CRFP (foveated VSR) and AdaGlimpse/AME (active exploration) were not executed: their action spaces or released weights do not fit this fixed-grid interface without retraining. The manuscript must state this and compare against them conceptually and via the fixed-grid uncertainty baselines.',
                  'Manuscript rewrite, English editing and Elsevier IVC formatting remain necessary.']+
                 ([] if complete_paper else ['The complete paper profile on natural videos with neural baselines has not been executed in this run.']))
    atomic_json(out/'readiness.json',readiness)
    archive=root/(out.name+'_evidence.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for p in sorted(out.rglob('*')):
            rel=p.relative_to(out)
            if p.is_file() and 'checkpoints' not in rel.parts:
                z.write(p,'results/'+str(rel))
        for p in sorted(Path(__file__).parent.glob('*.py')):z.write(p,'source/'+p.name)
        for folder in sorted((root/'official_models').glob('*')) if (root/'official_models').exists() else []:
            for p in folder.glob('*'):
                if p.suffix in ['.json','.py'] or p.name=='LICENSE':z.write(p,'official_models/'+folder.name+'/'+p.name)
    print('FINISHED:',archive,flush=True)
    return dict(output=str(out),archive=str(archive),summary=summary,paired=paired,controls=controls,
                tables=tables,readiness=readiness,sequence_results=df,backbone=backbone)
'''
(SOURCE_DIR / 'r9_runner.py').write_text(SOURCE, encoding='utf-8')
print('Wrote', 'r9_runner.py')

## 11. Validation (must pass before any experiment)

In [ ]:
import importlib, json

for module in ["video_core", "r9_core", "r9_data", "r9_neural", "r9_vsr_arch", "r9_engine", "r9_stats", "r9_checks", "r9_runner"]:
    sys.modules.pop(module, None)
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
importlib.invalidate_caches()

from r9_checks import (mathematical_checks, information_boundary_checks, coverage_simulation,
                       dataset_reader_checks, baseline_interface_checks)
from r9_runner import RunConfig, run_pipeline
import cv2
cv2.setNumThreads(THREADS)

CHECKS = {}
for check in [mathematical_checks, dataset_reader_checks, information_boundary_checks,
              coverage_simulation, baseline_interface_checks]:
    CHECKS[check.__name__] = check()
    print(check.__name__, "passed")
(WORKDIR / "implementation_checks.json").write_text(json.dumps(CHECKS, indent=2, default=str))

## 12. Source split (DAVIS roles from earlier revisions, kept fixed)

In [ ]:
LEGACY_SPLIT = {'fit': ['boat', 'boxing-fisheye', 'breakdance-flare', 'bus', 'car-turn', 'cat-girl', 'classic-car', 'crossing', 'dog-agility', 'dogs-scale', 'drift-turn', 'drone', 'elephant', 'flamingo', 'hike', 'hockey', 'horsejump-low', 'kid-football', 'kite-walk', 'koala', 'lady-running', 'lindy-hop', 'lucia', 'mallard-fly', 'mallard-water', 'miami-surf', 'motocross-bumps', 'paragliding', 'planes-water', 'scooter-board', 'scooter-gray', 'sheep', 'skate-park', 'snowboard', 'swing', 'train', 'tuk-tuk', 'upside-down', 'varanus-cage', 'walking'], 'development': ['bear', 'bmx-bumps', 'color-run', 'dance-jump', 'night-race', 'rallye', 'rhino', 'rollerblade', 'soccerball', 'tennis'], 'reserved': ['dancing', 'disc-jockey', 'dog-gooses', 'longboard', 'motorbike', 'schoolgirls', 'stroller', 'stunt', 'surf', 'tractor-sand'], 'test': ['bike-packing', 'blackswan', 'bmx-trees', 'breakdance', 'camel', 'car-roundabout', 'car-shadow', 'cows', 'dance-twirl', 'dog', 'dogs-jump', 'drift-chicane', 'drift-straight', 'goat', 'gold-fish', 'horsejump-high', 'india', 'judo', 'kite-surf', 'lab-coat', 'libby', 'loading', 'mbike-trick', 'motocross-jump', 'paragliding-launch', 'parkour', 'pigs', 'scooter-black', 'shooting', 'soapbox']}

## 13. Run

The first lines printed show the locked plan and the number of frame-method evaluations. Re-running this cell after a disconnect resumes from the checkpoints.

In [ ]:
CONFIG = RunConfig(
    root=str(WORKDIR),
    data_root=str(LOCAL_DATA),
    profile=PROFILE,
    experiment_id=EXPERIMENT_ID,
    run_neural=RUN_NEURAL,
    include_video_sr=INCLUDE_VIDEO_SR,
    device=DEVICE,
    download_data=DOWNLOAD_DATA,
    dataset_roots=DATASET_ROOTS,
    threads=THREADS,
    workers=WORKERS,
    neural_workers=NEURAL_WORKERS,
    neural_threads=NEURAL_THREADS,
    export_figures=EXPORT_FIGURES,
    line_dpi=LINE_DPI,
    high_resolution_reds=HIGH_RESOLUTION_REDS,
    save_all_traces=SAVE_ALL_TRACES,
)

RESULT = run_pipeline(CONFIG, LEGACY_SPLIT)

## 14. Results

In [ ]:
from IPython.display import display, Image, FileLink, Markdown
import pandas as pd

OUT = Path(RESULT["output"])
display(pd.DataFrame([RESULT["readiness"]]).T)
if (OUT / "baseline_sanity.json").exists():
    display(Markdown("**Baseline sanity gate (PSNR gain over bicubic, dB)**"))
    display(pd.read_json(OUT / "baseline_sanity.json").T)

tables = RESULT["tables"]
for name in sorted(tables):
    if name.startswith(("table_main", "table_reds_train_full", "table_reds_train_detail", "table_neural", "table_latency")):
        display(Markdown(f"**{name}**"))
        display(tables[name].round(6))

paired = RESULT["paired"]
if len(paired):
    display(Markdown("**Paired comparisons: ours vs each baseline (positive favours ours)**"))
    display(paired.query("study in ['main','reds_train_full','reds_train_detail','neural_native','neural_sensor'] and metric in ['psnr_db','augmented_pqv','write_pqv']")[[
        "study","dataset","regime","baseline_display","metric","effect_type",
        "reduction_pct","ci_low","ci_high","holm_p"]].round(5))

backbone = RESULT["backbone"]
if len(backbone):
    display(Markdown("**Same-backbone gaze comparisons (added after the laptop pilot): backbone + TwinAudit gaze vs "
                     "backbone + memory / + patch (positive favours TwinAudit gaze)**"))
    display(backbone[["study","dataset","regime","target_display","baseline_display","metric","effect_type",
                      "reduction_pct","ci_low","ci_high","holm_p"]].round(5))

print("LaTeX tables:", OUT / "tables")
print("Evidence:", RESULT["archive"])
display(FileLink(RESULT["archive"]))

In [ ]:
if EXPORT_FIGURES:
    for pattern in [
        "00_graphical_abstract.png",
        "01_controlled_policy_main_*_nominal.png",
        "01_controlled_policy_reds_train_*_nominal.png",
        "02_published_backbones_neural_native_*.png",
        "05_online_audit.png",
        "07_twin_audit_reliability.png",
        "08_write_tradeoff_*_nominal.png",
    ]:
        for match in sorted((OUT / "figures").glob(pattern))[:2]:
            display(Image(filename=str(match), width=900))

In [ ]:
print("Evidence archive (results, tables, figures, source):", RESULT["archive"])
